# Subsystem reports and LLM-as-judge evaluation

Builds System 1/2/3 reports (JSON + Markdown) from one run's artifacts and chat logs
(`results_seedstudy/seed_101/Query1`), then judges each report with only that subsystem's
four rubric dimensions from `rubric_detailed.yaml`.

- Reports are written to `evaluation_trial/trial1/system{1,2,3}_report.{json,md}`.
- Each judge run saves its full prompt, raw response, and scores under
  `evaluation_trial/trial1/s{1,2,3}_evaluation/`.
- The evaluator system prompts live in the **Evaluator prompts** cell; edit them there
  to test prompt variants, then re-run the judge cell(s).
- The per-dimension rubric text comes from `evaluation_refined/rubric_detailed.yaml`;
  edit that file to change metric definitions.
- The older summary-document evaluation cycle (three subsystem summaries, all 12 metrics
  in one blind call) lives in `run_refined_evaluation.py` (CLI) and `run_seed101.sh`.

```bash
conda activate MARS
jupyter lab evaluation_refined/evaluate_summaries.ipynb
```

## Making the System 1 report (JSON + Markdown)

Builds `system1_report.json` and `system1_report.md` from the run's System 1 artifact and chat log, in this format:

1. the query
2. RAG results from steps 0-1 (initial retrieval on the query sentence)
3. each generated question: 3.1 the question, 3.2 the LLM answer, 3.3 sources of that answer (RAG document names, KG node names), 3.4 answered self-check (0 vs 1)
4. derived outputs: 4.1 material properties, 4.2 hard constraints

Note: the answered flag only gates whether extraction runs at all; the extraction prompts include ALL Q&A pairs, including not-answered ones.

In [1]:
from pathlib import Path
import json
import re

# ============================================================
# INPUT FILES (same run as the section above)
# ============================================================

SYSTEM1_JSON = Path(
    "/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1/artifacts/system1_2026090312_0.json"
)

SYSTEM1_CHAT_LOG = Path(
    "/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1/artifacts/chats/system1_chat_log_material_requirements_2026090312_0.json"
)

# ============================================================
# OUTPUT FILES
# ============================================================

REPORT_JSON = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system1_report.json"
)

REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system1_report.md"
)

_WS_RE = re.compile(r"\s+")
_TITLE_RE = re.compile(r"Metadata:\s*title:\s*(.+)")
_QUESTION_RE = re.compile(
    r"Question:\s*(.*?)\n\s*\nRetrieved Information",
    re.S,
)


def _norm(text):
    return _WS_RE.sub(" ", str(text or "")).strip()


def build_system1_report(system1_json_path, chat_log_path):
    """Build the report dict:

    1. query
    2. initial RAG retrieval (the analyze calls on the query sentence)
    3. per generated question: question text, LLM answer, sources
       (RAG document titles + KG node names), answered self-check (0/1)
    4. derived material properties and derived hard constraints
    """
    with open(system1_json_path, encoding="utf-8") as f:
        system1 = json.load(f)
    with open(chat_log_path, encoding="utf-8") as f:
        chat = json.load(f)

    query = _norm(system1.get("sentence", ""))

    initial_rag = []
    slots = []  # one per question, in log order: {"rag": ..., "kg": ...}
    questions = []

    for interaction in chat.get("interactions", []):
        kind = interaction.get("type")
        method = interaction.get("method")
        data = interaction.get("data", {}) or {}

        if kind == "rag" and method == "analyze":
            docs = [
                {
                    "id": r.get("id"),
                    "distance": round(r["distance"], 3)
                    if isinstance(r.get("distance"), float)
                    else r.get("distance"),
                    "preview": (r.get("content_preview") or "")[:200],
                }
                for r in (data.get("results") or [])
            ]
            entry = {
                "query": _norm(data.get("query")),
                "keyword_filter": data.get("keywords") or [],
                "num_documents": len(docs),
                "documents": docs,
            }
            # Retrieval on the input sentence itself is step 0-1; every
            # other analyze call belongs to a generated question.
            if entry["query"] == query:
                initial_rag.append(entry)
            else:
                slots.append({"rag": entry, "kg": None})

        elif kind == "kg" and method == "find_connections":
            # find_connections runs right after its question's RAG call,
            # so it attaches to the most recent question slot.
            if slots and slots[-1]["kg"] is None:
                slots[-1]["kg"] = data

        elif kind == "llm" and method == "answer_question":
            prompt = data.get("user_prompt", "") or ""
            match = _QUESTION_RE.search(prompt)
            question_text = _norm(match.group(1)) if match else ""

            # Document titles exactly as the LLM saw them in the prompt,
            # in order and WITHOUT dedup, so position k corresponds to the
            # answer's inline citation [Document k].
            titles = [t.strip() for t in _TITLE_RE.findall(prompt) if t.strip()]

            # Match this answer to its retrieval slot by question text,
            # not by position, so a skipped KG call cannot shift sources.
            slot = next(
                (s for s in slots if s["rag"]["query"] == question_text),
                None,
            )
            rag_ids, kg_nodes = [], []
            if slot is not None:
                rag_ids = [d["id"] for d in slot["rag"]["documents"]]
                if slot["kg"]:
                    kg_nodes = list(slot["kg"].get("matched_node_ids") or [])

            questions.append(
                {
                    "index": len(questions) + 1,
                    "question": question_text,
                    "answer": data.get("response", ""),
                    "sources": {
                        "rag_documents": titles or rag_ids,
                        "rag_document_ids": rag_ids,
                        "kg_nodes": kg_nodes,
                    },
                    "answered": None,
                }
            )

        elif (
            kind == "llm"
            and method == "_is_question_answered"
            and questions
        ):
            resp = str(data.get("response", "")).strip()
            questions[-1]["answered"] = (
                int(resp) if resp in ("0", "1") else None
            )

    return {
        "run": f"{chat.get('pipeline')}/{chat.get('run_id')}",
        "query": query,
        "initial_rag": initial_rag,
        "questions": questions,
        "derived_material_properties": system1.get("extracted_keywords", []) or [],
        "derived_hard_constraints": system1.get("extracted_constraints", []) or [],
    }


report = build_system1_report(SYSTEM1_JSON, SYSTEM1_CHAT_LOG)

REPORT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"questions: {len(report['questions'])}")
for q in report["questions"]:
    print(
        f"  Q{q['index']} answered={q['answered']} "
        f"rag_docs={len(q['sources']['rag_documents'])} "
        f"kg_nodes={len(q['sources']['kg_nodes'])}"
    )
print(
    f"properties: {len(report['derived_material_properties'])} | "
    f"constraints: {len(report['derived_hard_constraints'])}"
)
print(f"\nSaved JSON to:\n{REPORT_JSON}")


questions: 4
  Q1 answered=0 rag_docs=2 kg_nodes=5
  Q2 answered=1 rag_docs=2 kg_nodes=5
  Q3 answered=1 rag_docs=3 kg_nodes=4
  Q4 answered=1 rag_docs=4 kg_nodes=4
properties: 30 | constraints: 17

Saved JSON to:
/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system1_report.json


In [2]:
def _demote_headings(text, by=2):
    """Push the LLM answer's own headings below this report's #### level.

    The answer is pasted verbatim, so its ### subheadings would otherwise
    render at the same level as our ### Question N. Clamped at ######,
    Markdown's deepest heading.
    """
    out = []
    for line in str(text or "").splitlines():
        stripped = line.lstrip()
        if stripped.startswith("#"):
            hashes = len(stripped) - len(stripped.lstrip("#"))
            line = "#" * min(hashes + by, 6) + stripped.lstrip("#")
        out.append(line)
    return "\n".join(out)


def render_system1_report_md(report):
    """Render the report dict as Markdown, following the agreed format:

    1. the query
    2. RAG results from steps 0-1
    3. each generated question: 3.1 question, 3.2 LLM answer,
       3.3 sources (RAG document names, KG node names), 3.4 answered 0/1
    4.1 derived material properties, 4.2 derived hard constraints
    """
    lines = []

    lines.append(f"# System 1 Report - {report['run']}")
    lines.append("")

    lines.append("## 1. Query")
    lines.append("")
    lines.append(report["query"])
    lines.append("")

    lines.append("## 2. RAG results (steps 0-1)")
    for i, r in enumerate(report["initial_rag"], 1):
        keyword_filter = (
            "; ".join(r["keyword_filter"]) if r["keyword_filter"] else "none"
        )
        lines.append("")
        lines.append(
            f"RAG query {i} (keyword filter: {keyword_filter}): "
            f"{r['num_documents']} documents"
        )
        for d in r["documents"]:
            lines.append(
                f"- `{d['id']}` (distance {d['distance']}): {d['preview']}"
            )
    lines.append("")

    lines.append("## 3. Generated questions by LLM")
    for q in report["questions"]:
        lines.append("")
        lines.append(f"### Question {q['index']}")
        lines.append("")
        lines.append("#### 3.1 Question")
        lines.append("")
        lines.append(q["question"])
        lines.append("")
        lines.append("#### 3.2 LLM answer")
        lines.append("")
        lines.append(_demote_headings(q["answer"]))
        lines.append("")
        lines.append("#### 3.3 Sources")
        lines.append("")
        lines.append("RAG documents (numbered as cited in the answer, "
                      "e.g. [Document 2] = document 2 below):")
        for k, name in enumerate(q["sources"]["rag_documents"], 1):
            lines.append(f"{k}. {name}")
        lines.append("")
        kg_nodes = q["sources"]["kg_nodes"]
        lines.append(
            "KG nodes: " + (", ".join(kg_nodes) if kg_nodes else "none")
        )
        lines.append("")
        lines.append("#### 3.4 Answered (LLM self-check)")
        lines.append("")
        label = {1: "1 (answered)", 0: "0 (not answered)"}.get(
            q["answered"], "unknown"
        )
        lines.append(f"**{label}**")
        lines.append("")
        lines.append("---")
    lines.append("")

    lines.append("## 4. Derived outputs by LLM")
    lines.append("")
    lines.append("### 4.1 Derived material properties")
    lines.append("")
    for i, prop in enumerate(report["derived_material_properties"], 1):
        lines.append(f"{i}. {prop}")
    lines.append("")
    lines.append("### 4.2 Derived hard constraints")
    lines.append("")
    for i, constraint in enumerate(report["derived_hard_constraints"], 1):
        lines.append(f"{i}. {constraint}")
    lines.append("")

    return "\n".join(lines)


markdown_text = render_system1_report_md(report)

REPORT_MD.parent.mkdir(parents=True, exist_ok=True)
REPORT_MD.write_text(markdown_text, encoding="utf-8")

print(f"Saved Markdown to:\n{REPORT_MD}")


Saved Markdown to:
/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system1_report.md


## Making the System 2 report (JSON + Markdown)

Builds `system2_report.json` and `system2_report.md` from the run's System 2 artifact and chat log, in this format:

1. the extracted properties + hard constraints from System 1
2. the KG call mapping properties to materials (paths whose nodes become candidates)
3. LLM classification of the harvested path nodes (material-related or not; shows the batch where index-style answers were silently dropped by the parser)
4. LLM classification of the whole material-informed subgraph (material / property / neither)
5. per candidate iteration: 5.1 proposal + justification, 5.2 PFAS check with tie-break re-query, 5.3 keyword extraction, 5.3.1 the 6 validation questions, 5.3.2 answers with RAG sources (patents + materialdb doc ids), 5.4 question keywords for the KG lookup, 5.5 feasibility verdict with reasoning, 5.6 failure summary fed into the next proposal

The loop repeats until a candidate passes feasibility (iteration 1 ends at 5.2: PFAS-rejected candidates never reach RAG validation; the accepted final iteration has no 5.6).

In [3]:
from pathlib import Path
import json
import re

# ============================================================
# INPUT FILES (same run as the System 1 report above)
# ============================================================

SYSTEM2_JSON = Path(
    "/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1/artifacts/system2_2026090312_0.json"
)

SYSTEM2_CHAT_LOG = Path(
    "/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1/artifacts/chats/system2_chat_log_material_discovery_2026090312_0.json"
)

# ============================================================
# OUTPUT FILES
# ============================================================

S2_REPORT_JSON = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system2_report.json"
)

S2_REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system2_report.md"
)

_NODE_LINE_RE = re.compile(r"^\s*\d+\.\s*Node ID:\s*(.+?)(?:,\s*Attributes:.*)?\s*$")
_NAME_RE = re.compile(r"Material Name:\s*(.+)")
_FEASIBLE_RE = re.compile(r"FEASIBLE:\s*(YES|NO)", re.IGNORECASE)
_VIOLATED_RE = re.compile(r"CONSTRAINTS_VIOLATED:\s*(.+)")
_NUMQ_RE = re.compile(r"^\s*\d+\.\s+(.+?)\s*$")


def _nodes_from_prompt(user_prompt):
    """The classifier prompts list nodes as 'N. Node ID: <name>'."""
    return [
        m.group(1).strip()
        for line in (user_prompt or "").splitlines()
        if (m := _NODE_LINE_RE.match(line))
    ]


def _json_block(text):
    """Extract the JSON object from a classifier response (may be fenced)."""
    a, b = text.find("{"), text.rfind("}")
    if a == -1 or b <= a:
        return {}
    try:
        return json.loads(text[a : b + 1])
    except json.JSONDecodeError:
        return {}


def _parse_questions(text):
    out = []
    for line in (text or "").splitlines():
        m = _NUMQ_RE.match(line)
        if m:
            out.append(m.group(1).strip())
    return out


def build_system2_report(system2_json_path, chat_log_path):
    """Walk the System 2 chat log in order and reconstruct:

    1. S1 inputs (properties + hard constraints)
    2. the KG property-to-material mapping (paths + harvested nodes)
    3. path-node classification (material or not)
    4. whole-subgraph classification (material / property / neither)
    5. the propose -> PFAS-check -> validate loop, one entry per candidate
    """
    with open(system2_json_path, encoding="utf-8") as f:
        system2 = json.load(f)
    with open(chat_log_path, encoding="utf-8") as f:
        chat = json.load(f)

    report = {
        "run": f"{chat.get('pipeline')}/{chat.get('run_id')}",
        "duration_seconds": chat.get("duration_seconds"),
        "s1_inputs": {
            "properties": (system2.get("properties") or {}).get("required", []),
            "hard_constraints": system2.get("constraints", []),
        },
        "kg_property_mapping": None,
        "path_node_classification": {"batches": []},
        "subgraph_classification": {"material_nodes": [], "property_nodes": []},
        "iterations": [],
        "final_candidate": {
            "material_name": (system2.get("candidate") or {}).get("material_name"),
            "success": system2.get("success"),
            "iterations": system2.get("iterations"),
        },
        "material_classes": [
            mc.get("node_id")
            for mc in (system2.get("property_mapping") or {}).get("material_classes", [])
        ],
    }

    iteration = None
    pending_rag = []  # the (up to) two RAG calls preceding each answer_question

    for it in chat.get("interactions", []):
        kind, method = it.get("type"), it.get("method")
        data = it.get("data", {}) or {}
        resp = data.get("response", "") or ""
        prompt = data.get("user_prompt", "") or ""

        # ---- stage 2: the one logged KG record ----
        if kind == "kg" and method == "map_properties_to_materials":
            samples = []
            for path in data.get("found_paths_sample") or []:
                if isinstance(path, dict) and path.get("path"):
                    samples.append(" -> ".join(str(n) for n in path["path"]))
            report["kg_property_mapping"] = {
                "num_keywords": len(data.get("keywords") or []),
                "result_summary": data.get("result_summary"),
                "total_paths": data.get("total_paths"),
                "sample_paths": samples[:5],
            }
            continue

        # ---- stage 3: path-node classifier (material or not) ----
        if kind == "llm" and method == "_classify_nodes_batch":
            batch_nodes = _nodes_from_prompt(prompt)
            returned = _json_block(resp).get("material_nodes", [])
            accepted = [x for x in returned if x in batch_nodes]
            report["path_node_classification"]["batches"].append({
                "num_nodes_in": len(batch_nodes),
                "nodes_in": batch_nodes,
                "material_nodes_returned": returned,
                "accepted_by_parser": accepted,
                # returned-but-not-accepted entries were silently dropped
                # (e.g. the model answered with list indices, not node names)
                "num_dropped": len(returned) - len(accepted),
            })
            continue

        # ---- stage 4: whole-subgraph classifier ----
        if kind == "llm" and method == "classify_nodes_batch":
            parsed = _json_block(resp)
            report["subgraph_classification"]["material_nodes"] += parsed.get("material_nodes", [])
            report["subgraph_classification"]["property_nodes"] += parsed.get("property_nodes", [])
            continue

        # ---- stage 5: the candidate loop ----
        if kind == "llm" and method == "propose_candidate":
            m = _NAME_RE.search(resp)
            iteration = {
                "iteration": len(report["iterations"]) + 1,
                "candidate_name": m.group(1).strip() if m else "",
                "justification": resp,
                "pfas_checks": [],
                "rejected_as_pfas": False,
                "keywords_for_queries": None,
                "validation_questions": [],
                "question_answers": [],
                "keywords_for_feasibility": None,
                "feasibility": None,
                "rejection_lessons": None,
            }
            report["iterations"].append(iteration)
            pending_rag = []
            continue

        if iteration is None:
            continue

        if kind == "llm" and method == "check_pfas":
            verdict = resp.strip().upper()
            iteration["pfas_checks"].append({
                "response": verdict,
                "tiebreak_requery": "re-evaluate" in prompt,
            })
            if verdict.startswith("YES"):
                iteration["rejected_as_pfas"] = True
            continue

        if kind == "llm" and method == "_extract_keywords_from_text":
            if not iteration["validation_questions"]:
                iteration["keywords_for_queries"] = resp
            else:
                iteration["keywords_for_feasibility"] = resp
            continue

        if kind == "llm" and method == "generate_validation_queries":
            iteration["validation_questions"] = _parse_questions(resp)
            continue

        if kind == "rag" and method == "analyze":
            pending_rag.append({
                "query": data.get("query") or "",
                "doc_ids": [r.get("id") for r in (data.get("results") or [])],
            })
            continue

        if kind == "llm" and method == "answer_question":
            # the two preceding RAG calls belong to this question,
            # in MultiAnalyst wiring order: patents first, materialdb second
            rag_pair = pending_rag[-2:]
            pending_rag = []
            sources = {}
            if len(rag_pair) == 2:
                sources = {
                    "patents": rag_pair[0]["doc_ids"],
                    "materialdb": rag_pair[1]["doc_ids"],
                }
            elif rag_pair:
                sources = {"unknown": rag_pair[0]["doc_ids"]}
            iteration["question_answers"].append({
                "question": rag_pair[0]["query"] if rag_pair else "",
                "answer": resp,
                "rag_documents": sources,
            })
            continue

        if kind == "llm" and method == "validate_feasibility":
            fm = _FEASIBLE_RE.search(resp)
            vm = _VIOLATED_RE.search(resp)
            iteration["feasibility"] = {
                "feasible": (fm.group(1).upper() == "YES") if fm else None,
                "constraints_violated": vm.group(1).strip() if vm else None,
                "reasoning": resp,
            }
            continue

        if kind == "llm" and method == "summarize_rejection_lessons":
            iteration["rejection_lessons"] = resp
            continue

    return report


s2_report = build_system2_report(SYSTEM2_JSON, SYSTEM2_CHAT_LOG)

S2_REPORT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(S2_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(s2_report, f, indent=2, ensure_ascii=False)

print(f"properties: {len(s2_report['s1_inputs']['properties'])} | "
      f"constraints: {len(s2_report['s1_inputs']['hard_constraints'])}")
kg = s2_report["kg_property_mapping"] or {}
print(f"kg paths: {kg.get('total_paths')} | material classes: {len(s2_report['material_classes'])}")
for b in s2_report["path_node_classification"]["batches"]:
    print(f"  path-node batch: {b['num_nodes_in']} in, "
          f"{len(b['material_nodes_returned'])} returned, "
          f"{len(b['accepted_by_parser'])} accepted, {b['num_dropped']} dropped")
sc = s2_report["subgraph_classification"]
print(f"subgraph classes: {len(sc['material_nodes'])} material, {len(sc['property_nodes'])} property")
for itr in s2_report["iterations"]:
    f_ = itr["feasibility"]
    verdict = ("PFAS-rejected" if itr["rejected_as_pfas"]
               else ("FEASIBLE" if f_ and f_["feasible"] else "NOT feasible" if f_ else "no verdict"))
    print(f"  iter {itr['iteration']}: {itr['candidate_name'][:55]} -> {verdict} "
          f"({len(itr['question_answers'])} Q&A)")
print(f"\nSaved JSON to:\n{S2_REPORT_JSON}")

properties: 30 | constraints: 17
kg paths: 120 | material classes: 14
  path-node batch: 50 in, 28 returned, 0 accepted, 28 dropped
  path-node batch: 27 in, 14 returned, 14 accepted, 0 dropped
subgraph classes: 134 material, 53 property
  iter 1: PEEK‑FKM Hybrid Thermoplastic Elastomer -> PFAS-rejected (0 Q&A)
  iter 2: RILSAN® BMNO / Transparent Polyurethane Elastomer Therm -> NOT feasible (6 Q&A)
  iter 3: RILSAN® BMNO / Silica Nanoparticle Composite -> NOT feasible (6 Q&A)
  iter 4: ENGAGE™ 030902 HEALTH+ Polyolefin Elastomer / TiO2 Nano -> FEASIBLE (6 Q&A)

Saved JSON to:
/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system2_report.json


In [ ]:
def _s2_demote_headings(text, by=2):
    """Demote headings inside pasted LLM text (same as the System 1 report)."""
    out = []
    for line in str(text or "").splitlines():
        stripped = line.lstrip()
        if stripped.startswith("#"):
            hashes = len(stripped) - len(stripped.lstrip("#"))
            line = "#" * min(hashes + by, 6) + stripped.lstrip("#")
        out.append(line)
    return "\n".join(out)


_S2_MATDB_CACHE = {}

def _s2_load_material_db():
    """Internal lab-materials database (20 entries), for inventory records."""
    if "db" not in _S2_MATDB_CACHE:
        import json as _json, os as _os
        path = _os.environ.get(
            "MARS_MATERIAL_DB",
            "data/MARS_Data/MaterialDB_paper/internal_material_database.json")
        try:
            _S2_MATDB_CACHE["db"] = _json.load(open(path, encoding="utf-8"))
        except Exception:
            _S2_MATDB_CACHE["db"] = {}
    return _S2_MATDB_CACHE["db"]


def _s2_norm_name(s):
    import re as _re
    s = _re.sub(r"[™®℠]", " ", str(s))
    # suppliers also write trademarks as literal ASCII suffixes: "EdgetekTM"
    s = _re.sub(r"(?<=[a-zA-Z])(?:TM|\(TM\)|\(R\))(?=\s|$)", " ", s)
    return _re.sub(r"\s+", " ", s).strip().lower()


def _s2_norm_dashes(s):
    """LLM text uses typographic dashes (e.g. U+2011) inside catalog IDs."""
    import re as _re
    return _re.sub(r"[‐‑‒–—―]", "-", str(s))


def _s2_inventory_records(justification_text):
    """Match proposed components to internal-database entries by catalog ID
    (preferred) or product name, and render a property excerpt for each, so
    database anchoring is verifiable inside the report."""
    db = _s2_load_material_db()
    if not db:
        return []
    text = _s2_norm_dashes(justification_text or "")
    text_norm = _s2_norm_name(text)
    seen, lines = set(), []
    for key, entry in db.items():
        eid = entry.get("id") or key
        name = entry.get("name") or key
        base = eid[4:] if eid.startswith("the-") else eid
        matched = (eid in text) or (base in text) or (_s2_norm_name(name) in text_norm)
        canon = base
        if not matched or canon in seen:
            continue
        seen.add(canon)
        lines.append(f"**Inventory record `{eid}`** — {name} ({entry.get('supplier', 'unknown supplier')})")
        lines.append("")
        ex = entry.get("extracted") or {}
        shown = 0
        for k, v in ex.items():
            if shown >= 15:
                lines.append(f"- ... ({len(ex) - shown} further datasheet fields omitted)")
                break
            vs = ", ".join(str(x) for x in v) if isinstance(v, list) else str(v)
            if len(vs) > 220:
                vs = vs[:220] + "..."
            lines.append(f"- {k}: {vs}")
            shown += 1
        lines.append("")
    return lines


def render_system2_report_md(report):
    """Render the System 2 report in the agreed format:

    1. extracted properties + hard constraints from System 1
    2. KG call harvesting path nodes related to the properties
    3. LLM sorts path nodes into material-related or not
    4. LLM sorts the whole subgraph into material / property / neither
    5. per candidate: proposal, PFAS check (with tie-break), keywords,
       6 validation questions with RAG-sourced answers, feasibility
       verdict, and the failure summary after a rejection
    """
    lines = []
    lines.append(f"# System 2 Report - {report['run']}")
    lines.append("")

    lines.append("## 1. Inputs from System 1")
    lines.append("")
    lines.append("### 1.1 Extracted properties")
    lines.append("")
    for i, p in enumerate(report["s1_inputs"]["properties"], 1):
        lines.append(f"{i}. {p}")
    lines.append("")
    lines.append("### 1.2 Hard constraints")
    lines.append("")
    for i, c in enumerate(report["s1_inputs"]["hard_constraints"], 1):
        lines.append(f"{i}. {c}")
    lines.append("")

    kg = report.get("kg_property_mapping") or {}
    lines.append("## 2. KG call: property-to-material mapping")
    lines.append("")
    rs = kg.get("result_summary") or {}
    lines.append(f"- Properties mapped: {rs.get('num_properties')}")
    lines.append(f"- Paths found between mapped nodes: {kg.get('total_paths')}")
    lines.append(f"- Material classes out: {rs.get('num_material_classes')}")
    lines.append("")
    lines.append("Sample paths (nodes on these paths become classification candidates):")
    lines.append("")
    for s in kg.get("sample_paths", []):
        lines.append(f"- `{s}`")
    lines.append("")

    lines.append("## 3. LLM classification of the harvested path nodes")
    lines.append("")
    total_in = sum(b["num_nodes_in"] for b in report["path_node_classification"]["batches"])
    lines.append(f"{total_in} path nodes classified in "
                 f"{len(report['path_node_classification']['batches'])} batches "
                 f"(material-related or not).")
    lines.append("")
    for bi, b in enumerate(report["path_node_classification"]["batches"], 1):
        lines.append(f"### Batch {bi}: {b['num_nodes_in']} nodes in")
        lines.append("")
        lines.append(f"- Returned as material: {len(b['material_nodes_returned'])}")
        lines.append(f"- Accepted by the parser: {len(b['accepted_by_parser'])}"
                     + (f" ({b['num_dropped']} dropped: the model answered with"
                        " list indices instead of node names)" if b["num_dropped"] else ""))
        if b["accepted_by_parser"]:
            lines.append(f"- Accepted nodes: {', '.join(b['accepted_by_parser'])}")
        lines.append("")

    sc = report["subgraph_classification"]
    lines.append("## 4. LLM classification of the whole material-informed subgraph")
    lines.append("")
    lines.append(f"- Material-related nodes ({len(sc['material_nodes'])}): "
                 + ", ".join(sc["material_nodes"]))
    lines.append("")
    lines.append(f"- Property-related nodes ({len(sc['property_nodes'])}): "
                 + ", ".join(sc["property_nodes"]))
    lines.append("")

    lines.append("## 5. Candidate material proposal and verification")
    lines.append("")
    for itr in report["iterations"]:
        lines.append(f"### Iteration {itr['iteration']}: {itr['candidate_name']}")
        lines.append("")
        lines.append("#### 5.1 Proposed candidate and justification")
        lines.append("")
        lines.append(_s2_demote_headings(itr["justification"]))
        lines.append("")

        inv = _s2_inventory_records(itr["justification"])
        if inv:
            lines.append("#### 5.1.1 Internal-database records for proposed components")
            lines.append("")
            lines.append("Datasheet excerpts from the lab-materials inventory for every "
                         "component of this proposal that matches a database entry "
                         "(by catalog ID or product name); use them to check the "
                         "proposal's identity and property claims.")
            lines.append("")
            lines.extend(inv)

        lines.append("#### 5.2 PFAS check")
        lines.append("")
        for ci, chk in enumerate(itr["pfas_checks"], 1):
            kind = "tie-break re-query" if chk["tiebreak_requery"] else "first opinion"
            lines.append(f"- Check {ci} ({kind}): **{chk['response']}**")
        lines.append("")
        if itr["rejected_as_pfas"]:
            lines.append("**Rejected as PFAS - iteration ends here (no RAG validation).**")
            lines.append("")

        if itr["keywords_for_queries"]:
            lines.append("#### 5.3 Keywords extracted for the proposed candidate material")
            lines.append("")
            lines.append(_s2_demote_headings(itr["keywords_for_queries"]))
            lines.append("")

        if itr["validation_questions"]:
            lines.append("#### 5.3.1 Generated validation questions")
            lines.append("")
            for qi, q in enumerate(itr["validation_questions"], 1):
                lines.append(f"{qi}. {q}")
            lines.append("")

        if itr["question_answers"]:
            lines.append("#### 5.3.2 Answers (RAG on materialdb + patents)")
            lines.append("")
            for qi, qa in enumerate(itr["question_answers"], 1):
                lines.append(f"**Q{qi}. {qa['question']}**")
                lines.append("")
                lines.append(_s2_demote_headings(qa["answer"]))
                lines.append("")
                for src, ids in (qa.get("rag_documents") or {}).items():
                    shown = ", ".join(f"`{i}`" for i in ids)
                    lines.append(f"- {src} documents: {shown}")
                lines.append("")

        if itr["keywords_for_feasibility"]:
            lines.append("#### 5.4 Keywords from the validation questions (for KG lookup)")
            lines.append("")
            lines.append(_s2_demote_headings(itr["keywords_for_feasibility"]))
            lines.append("")

        if itr["feasibility"]:
            f_ = itr["feasibility"]
            verdict = "YES" if f_["feasible"] else "NO"
            lines.append("#### 5.5 Feasibility check against properties + hard constraints")
            lines.append("")
            lines.append(f"**FEASIBLE: {verdict}**")
            if f_.get("constraints_violated"):
                lines.append("")
                lines.append(f"Constraints violated: {f_['constraints_violated']}")
            lines.append("")
            lines.append(_s2_demote_headings(f_["reasoning"]))
            lines.append("")

        if itr["rejection_lessons"]:
            lines.append("#### 5.6 Failure summary (fed into the next proposal)")
            lines.append("")
            lines.append(_s2_demote_headings(itr["rejection_lessons"]))
            lines.append("")

        lines.append("---")
        lines.append("")

    fc = report["final_candidate"]
    lines.append("## Outcome")
    lines.append("")
    lines.append(f"Accepted candidate after {fc.get('iterations')} iterations: "
                 f"**{fc.get('material_name')}**")
    lines.append("")
    return "\n".join(lines)


s2_markdown = render_system2_report_md(s2_report)

S2_REPORT_MD.parent.mkdir(parents=True, exist_ok=True)
S2_REPORT_MD.write_text(s2_markdown, encoding="utf-8")

print(f"Saved Markdown to:\n{S2_REPORT_MD}")


## Making the System 3 report (JSON + Markdown)

Builds `system3_report.json` and `system3_report.md` from the run's System 3 chat log, in this format:

1. proposed candidate and justification handed over by System 2
2. LLM decides whether the candidate is a single material or a composite (constituents, combination modes)
3. LLM-generated manufacturing process queries; for each, the RAG searches over manufacturing textbooks, materialdb, and patents are merged into one Evidence list
4. the four LLM-generated feasibility questions; for each, 4.1 the merged 3-source Evidence, 4.2 the LLM answer with confidence and cited documents
5. the feasibility verdict (feasible flag, blocking constraints, feedback to System 2)
6. the detailed process recipe with the 30-document backing evidence pack from the recipe prompt

In [5]:
from pathlib import Path
import json
import re

# ============================================================
# INPUT FILES (same run as the System 1/2 reports above)
# ============================================================

SYSTEM3_CHAT_LOG = Path(
    "/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1/artifacts/chats/system3_chat_log_manufacturability_assessment_2026090312_0.json"
)

# ============================================================
# OUTPUT FILES
# ============================================================

S3_REPORT_JSON = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system3_report.json"
)

S3_REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system3_report.md"
)

# System 3's MultiAnalyst queries its sources in this wiring order,
# so consecutive RAG calls with the same query text map to these labels.
S3_RAG_SOURCES = ["manufacturing_textbooks", "patents", "materialdb"]

_CAND_RE = re.compile(r"Candidate material:\s*(.+)")
_JUST_RE = re.compile(
    r"Candidate justification:\s*(.*?)\nUser constraints:", re.S
)
_DOC_HDR_RE = re.compile(r"\[Document \d+\]\s*\nSource:\s*(\S+)\s*\nID:\s*(\S+)")
_DOC_NUM_RE = re.compile(r"\[Document (\d+)\]\s*\nSource:\s*(\S+)\s*\nID:\s*(\S+)")


def _json_block(text):
    a, b = text.find("{"), text.rfind("}")
    if a == -1 or b <= a:
        return {}
    try:
        return json.loads(text[a : b + 1])
    except json.JSONDecodeError:
        return {}


def build_system3_report(chat_log_path):
    """Walk the System 3 chat log in order and reconstruct:

    1. the candidate + justification handed over by System 2
    2. the single-vs-composite decomposition
    3. the manufacturing process queries, each with merged 3-source evidence
    4. the four feasibility questions, each with merged evidence + answer
    5. the feasibility verdict
    6. the process recipe with its backing evidence pack
    """
    with open(chat_log_path, encoding="utf-8") as f:
        chat = json.load(f)

    report = {
        "run": f"{chat.get('pipeline')}/{chat.get('run_id')}",
        "duration_seconds": chat.get("duration_seconds"),
        "candidate": {},
        "decomposition": {},
        "process_queries": [],
        "feasibility_questions": [],
        "verdict": {},
        "recipe": {"steps": [], "backing_evidence": []},
    }

    pending_rag = []   # consecutive analyze calls not yet assigned
    phase = "process"  # switches to "feasibility" after the questions call

    def _merged_evidence(rag_calls):
        """Merge a 3-source triple into one evidence list."""
        merged = []
        for slot, call in enumerate(rag_calls):
            source = (
                S3_RAG_SOURCES[slot]
                if slot < len(S3_RAG_SOURCES) else "unknown"
            )
            for r in call["results"]:
                merged.append({
                    "source": source,
                    "id": r.get("id"),
                    "distance": round(r["distance"], 3)
                    if isinstance(r.get("distance"), float) else r.get("distance"),
                })
        return merged

    def _flush_rag_to(target_list):
        """Assign buffered RAG calls, three per query, in order."""
        for start in range(0, len(pending_rag), 3):
            triple = pending_rag[start : start + 3]
            idx = start // 3
            if idx < len(target_list):
                target_list[idx]["evidence"] = _merged_evidence(triple)

    for it in chat.get("interactions", []):
        kind, method = it.get("type"), it.get("method")
        data = it.get("data", {}) or {}
        resp = data.get("response", "") or ""
        prompt = data.get("user_prompt", "") or ""

        if kind == "llm" and method == "extract_material_constituents_for_manufacturing":
            m_name = _CAND_RE.search(prompt)
            m_just = _JUST_RE.search(prompt)
            report["candidate"] = {
                "material_name": m_name.group(1).strip() if m_name else "",
                "justification": m_just.group(1).strip() if m_just else "",
            }
            report["decomposition"] = _json_block(resp)

        elif kind == "llm" and method == "generate_decomposition_process_queries":
            for q in _json_block(resp).get("queries", []):
                report["process_queries"].append({
                    "query": q.get("query", ""),
                    "query_type": q.get("query_type", ""),
                    "constituent": q.get("constituent", ""),
                    "evidence": [],
                })

        elif kind == "rag" and method == "analyze":
            pending_rag.append({
                "query": data.get("query") or "",
                "results": data.get("results") or [],
            })

        elif kind == "llm" and method == "generate_feasibility_questions":
            _flush_rag_to(report["process_queries"])
            pending_rag = []
            phase = "feasibility"
            for q in _json_block(resp).get("questions", []):
                report["feasibility_questions"].append({
                    "question": q,
                    "evidence": [],
                    "answer": None,
                    "confidence": None,
                    "evidence_used": [],
                })

        elif kind == "llm" and method == "answer_feasibility_question":
            # the buffered triple belongs to the next unanswered question
            idx = next(
                (i for i, q in enumerate(report["feasibility_questions"])
                 if q["answer"] is None),
                None,
            )
            if idx is not None:
                report["feasibility_questions"][idx]["evidence"] = (
                    _merged_evidence(pending_rag[-3:])
                )
                # the answer prompt numbers its documents [Document N]; keep
                # that exact numbering so 'Evidence cited: Document N' resolves
                report["feasibility_questions"][idx]["numbered_evidence"] = [
                    {"n": int(n), "source": src, "id": i}
                    for n, src, i in _DOC_NUM_RE.findall(prompt)
                ]
                parsed = _json_block(resp)
                report["feasibility_questions"][idx].update({
                    "answer": parsed.get("answer", resp),
                    "confidence": parsed.get("confidence"),
                    "evidence_used": parsed.get("evidence_used", []),
                })
            pending_rag = []

        elif kind == "llm" and method == "assess_manufacturability_feasibility":
            report["verdict"] = _json_block(resp)

        elif kind == "llm" and method == "synthesize_process_recipe":
            report["recipe"]["steps"] = _json_block(resp).get("process_recipe", [])
            # the recipe prompt carries its own 30-document evidence pack
            report["recipe"]["backing_evidence"] = [
                {"n": int(n), "source": s, "id": i}
                for n, s, i in _DOC_NUM_RE.findall(prompt)
            ]

    return report


s3_report = build_system3_report(SYSTEM3_CHAT_LOG)

S3_REPORT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(S3_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(s3_report, f, indent=2, ensure_ascii=False)

print(f"candidate: {s3_report['candidate'].get('material_name')}")
dec = s3_report["decomposition"]
print(f"composite: {dec.get('is_composite')} | constituents: {dec.get('constituents')}")
print(f"process queries: {len(s3_report['process_queries'])} "
      f"(evidence docs per query: "
      f"{[len(q['evidence']) for q in s3_report['process_queries']]})")
for i, q in enumerate(s3_report["feasibility_questions"], 1):
    print(f"  feasibility Q{i}: confidence={q['confidence']} "
          f"evidence={len(q['evidence'])} docs")
print(f"verdict: feasible={s3_report['verdict'].get('feasible')} | "
      f"blocking={len(s3_report['verdict'].get('blocking_constraints', []))}")
print(f"recipe steps: {len(s3_report['recipe']['steps'])} | "
      f"backing evidence docs: {len(s3_report['recipe']['backing_evidence'])}")
print(f"\nSaved JSON to:\n{S3_REPORT_JSON}")


candidate: ENGAGE™ 030902 HEALTH+ Polyolefin Elastomer / TiO2 Nanoparticle Composite
composite: True | constituents: ['ENGAGE™ 030902 HEALTH+ Polyolefin Elastomer', 'TiO₂ Nanoparticles (anatase, ~20\u202fnm, 2\u202fwt%)']
process queries: 13 (evidence docs per query: [24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24])
  feasibility Q1: confidence=medium evidence=24 docs
  feasibility Q2: confidence=medium evidence=24 docs
  feasibility Q3: confidence=medium evidence=24 docs
  feasibility Q4: confidence=medium evidence=24 docs
verdict: feasible=True | blocking=0
recipe steps: 10 | backing evidence docs: 30

Saved JSON to:
/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system3_report.json


In [6]:
def _s3_demote_headings(text, by=2):
    """Demote headings inside pasted LLM text (same as the other reports)."""
    out = []
    for line in str(text or "").splitlines():
        stripped = line.lstrip()
        if stripped.startswith("#"):
            hashes = len(stripped) - len(stripped.lstrip("#"))
            line = "#" * min(hashes + by, 6) + stripped.lstrip("#")
        out.append(line)
    return "\n".join(out)


def _s3_evidence_lines(evidence):
    """One merged Evidence list, grouped by source."""
    lines = []
    by_source = {}
    for e in evidence:
        by_source.setdefault(e["source"], []).append(e)
    for source, docs in by_source.items():
        shown = ", ".join(f"`{d['id']}` ({d['distance']})" for d in docs)
        lines.append(f"- {source}: {shown}")
    return lines


def render_system3_report_md(report):
    """Render the System 3 report in the agreed format:

    1. proposed candidate and justification from System 2
    2. single-vs-composite decomposition
    3. manufacturing queries, each with merged 3-source Evidence
    4. four feasibility questions, each with merged Evidence + LLM answer
    5. feasibility verdict
    6. the process recipe with backing evidence
    """
    lines = []
    lines.append(f"# System 3 Report - {report['run']}")
    lines.append("")

    lines.append("## 1. Proposed candidate and justification from System 2")
    lines.append("")
    lines.append(f"**{report['candidate'].get('material_name', '')}**")
    lines.append("")
    lines.append(_s3_demote_headings(report["candidate"].get("justification", "")))
    lines.append("")

    dec = report.get("decomposition") or {}
    lines.append("## 2. Single material or composite?")
    lines.append("")
    kind = "composite" if dec.get("is_composite") else "single material"
    lines.append(f"The candidate is a **{kind}**.")
    lines.append("")
    lines.append("Constituents:")
    for c in dec.get("constituents", []):
        lines.append(f"- {c}")
    lines.append("")
    if dec.get("composition_notes"):
        lines.append(f"Composition notes: {dec['composition_notes']}")
        lines.append("")
    if dec.get("combination_modes"):
        lines.append("Combination modes: " + ", ".join(dec["combination_modes"]))
        lines.append("")

    lines.append("## 3. Manufacturing process queries")
    lines.append("")
    lines.append(f"{len(report['process_queries'])} queries generated by the LLM; "
                 "each one is searched over the manufacturing textbooks, "
                 "materialdb, and patents collections (merged below as Evidence).")
    lines.append("")
    for qi, q in enumerate(report["process_queries"], 1):
        tag = q["query_type"]
        if q.get("constituent"):
            tag += f", {q['constituent']}"
        lines.append(f"### Query {qi} ({tag})")
        lines.append("")
        lines.append(q["query"])
        lines.append("")
        lines.append("Evidence:")
        lines.append("")
        lines += _s3_evidence_lines(q.get("evidence", []))
        lines.append("")

    lines.append("## 4. Feasibility questions")
    lines.append("")
    for qi, q in enumerate(report["feasibility_questions"], 1):
        lines.append(f"### Question {qi}")
        lines.append("")
        lines.append(q["question"])
        lines.append("")
        lines.append("#### 4.1 Evidence")
        lines.append("")
        if q.get("numbered_evidence"):
            lines.append("Documents numbered exactly as cited in the answer "
                         "(e.g. 'Document 21' = item 21 below):")
            lines.append("")
            for d in q["numbered_evidence"]:
                lines.append(f"{d['n']}. {d['source']}: `{d['id']}`")
        else:
            lines += _s3_evidence_lines(q.get("evidence", []))
        lines.append("")
        lines.append("#### 4.2 LLM answer")
        lines.append("")
        lines.append(_s3_demote_headings(q.get("answer") or ""))
        lines.append("")
        meta = f"Confidence: **{q.get('confidence')}**"
        if q.get("evidence_used"):
            meta += " | Evidence cited: " + ", ".join(q["evidence_used"])
        lines.append(meta)
        lines.append("")

    v = report.get("verdict") or {}
    lines.append("## 5. Feasibility verdict")
    lines.append("")
    lines.append(f"**Feasible: {'YES' if v.get('feasible') else 'NO'}**")
    lines.append("")
    blocking = v.get("blocking_constraints") or []
    if blocking:
        lines.append("Blocking constraints:")
        for b in blocking:
            lines.append(f"- [{b.get('type')}/{b.get('severity')}] {b.get('description')}")
    else:
        lines.append("Blocking constraints: none listed.")
    lines.append("")
    if v.get("feedback_to_system2"):
        lines.append(f"Feedback to System 2: {v['feedback_to_system2']}")
        lines.append("")

    lines.append("## 6. Process recipe")
    lines.append("")
    for step in report["recipe"]["steps"]:
        lines.append(f"### Step {step.get('step_index')}: {step.get('description', '')}")
        lines.append("")
        cond = step.get("conditions")
        if isinstance(cond, dict):
            for k, val in cond.items():
                lines.append(f"- {k}: {val}")
        elif cond:
            lines.append(f"- Conditions: {cond}")
        if step.get("equipment_class"):
            lines.append(f"- Equipment: {step['equipment_class']}")
        if step.get("inputs"):
            lines.append(f"- Inputs: {', '.join(step['inputs'])}")
        lines.append("")

    lines.append("### Backing evidence for the recipe")
    lines.append("")
    be = report["recipe"]["backing_evidence"]
    if be and be[0].get("n") is not None:
        lines.append("Documents numbered exactly as presented to the recipe "
                     "synthesis step ([Document N]):")
        lines.append("")
        for e in be:
            lines.append(f"{e['n']}. {e['source']}: `{e['id']}`")
    else:
        by_source = {}
        for e in be:
            by_source.setdefault(e["source"], []).append(e["id"])
        for source, ids in by_source.items():
            lines.append(f"- {source} ({len(ids)}): " + ", ".join(f"`{i}`" for i in ids))
    lines.append("")
    return "\n".join(lines)


s3_markdown = render_system3_report_md(s3_report)

S3_REPORT_MD.parent.mkdir(parents=True, exist_ok=True)
S3_REPORT_MD.write_text(s3_markdown, encoding="utf-8")

print(f"Saved Markdown to:\n{S3_REPORT_MD}")


Saved Markdown to:
/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system3_report.md


## Evaluator prompts (edit here to test variants)

The next cell defines the three evaluator SYSTEM prompts as plain strings. Each judge cell
below uses its variable verbatim and rebuilds the USER prompt from the query, that system's
`rubric_s{1,2,3}_v2.yaml` (0-10 scale, evaluation principles, output schema), and the report
Markdown. To test a variant: edit the prompt here (or the rubric YAML), re-run this cell,
then re-run the judge cell; the exact prompt sent is saved as `judge_prompt_<ts>.txt` next
to the scores.

In [7]:
# ============================================================
# EVALUATOR SYSTEM PROMPTS - edit these to test variants.
# The three judge cells below read these variables verbatim.
# The per-dimension rubric text, principles, and output schema are
# loaded separately from rubric_s{1,2,3}_v2.yaml and appended into
# the user prompt.
# ============================================================

S1_JUDGE_SYSTEM_PROMPT = """\
You are an expert materials scientist acting as an evaluator. \
You will receive the SYSTEM 1 output of a material-substitution pipeline: \
a report containing the input query, the initial retrieval, each research \
question with the LLM answer, that answer's sources (RAG document names \
and knowledge-graph node names) and a 0/1 answered self-check, and the \
derived material properties and hard constraints.

Evaluate ONLY the System 1 criteria in the rubric, using the score scale defined in the rubric.

IMPORTANT:
- Be critical and specific; cite the exact items, numbers, or sources \
you relied on.
- Bracketed markers such as [Document 3] inside an answer are evidence \
citations: they refer to that question's numbered Sources list. An \
answer's statement that the documents do NOT contain a value is \
negative evidence.
- The 'Derived outputs by LLM' section is the product being judged; the \
Q&A section is its provenance. A derived number that appears in no \
answer and no document is unsupported.
- Do NOT evaluate candidate discovery or manufacturability; they are \
downstream systems and absent from this report.

You MUST respond with a valid JSON object and nothing else."""

S2_JUDGE_SYSTEM_PROMPT = """\
You are an expert materials scientist acting as an evaluator. \
You will receive the SYSTEM 2 output of a material-substitution pipeline: \
a report containing the System 1 requirement inputs (properties and hard \
constraints), the knowledge-graph grounding steps, and every candidate \
iteration: the proposal with justification, the PFAS checks, the \
validation questions, the retrieved-evidence answers (with source \
document ids), the feasibility verdict with reasoning, and the failure \
summaries between iterations.

Evaluate ONLY the System 2 criteria in the rubric, using the score scale defined in the rubric.

IMPORTANT:
- Be critical and specific; cite the exact iterations, claims, document \
ids, or verdict text you relied on.
- The final accepted candidate is the primary object being judged; the \
rejected iterations are evidence of the selection process's quality.
- A quantitative claim in a proposal or answer counts as supported only \
if the report shows retrieved text stating it; requirement numbers \
echoed back as candidate properties are unsupported.
- Do NOT evaluate System 1's requirement quality or System 3's \
manufacturability; they are outside this report's scope.

You MUST respond with a valid JSON object and nothing else."""

S3_JUDGE_SYSTEM_PROMPT = """\
You are an expert materials scientist and process engineer acting as an \
evaluator. You will receive the SYSTEM 3 output of a material-substitution \
pipeline: a manufacturability report containing the candidate handed over \
by System 2 with its justification, the single-vs-composite decomposition, \
the manufacturing process queries with their retrieved evidence \
(manufacturing textbooks, materialdb, patents), the four feasibility \
questions with evidence and LLM answers, the feasibility verdict, and the \
proposed process recipe with its backing evidence.

Evaluate ONLY the System 3 criteria in the rubric, using the score scale defined in the rubric.

IMPORTANT:
- Be critical and specific; cite the exact steps, conditions, question \
answers, or document ids you relied on.
- The process recipe and feasibility assessment are the primary objects \
being judged; take the candidate's stated composition as given.
- A processing condition counts as supported only if the report's \
evidence or answers state it; check the recipe's numbers against the \
Q&A answers for internal consistency.
- Do NOT evaluate System 1's requirement quality or System 2's \
candidate choice; they are outside this report's scope.

You MUST respond with a valid JSON object and nothing else."""

print("Evaluator prompts loaded:")
for name, txt in [("S1", S1_JUDGE_SYSTEM_PROMPT),
                  ("S2", S2_JUDGE_SYSTEM_PROMPT),
                  ("S3", S3_JUDGE_SYSTEM_PROMPT)]:
    print(f"  {name}_JUDGE_SYSTEM_PROMPT: {len(txt)} chars")


Evaluator prompts loaded:
  S1_JUDGE_SYSTEM_PROMPT: 952 chars
  S2_JUDGE_SYSTEM_PROMPT: 1387 chars
  S3_JUDGE_SYSTEM_PROMPT: 1262 chars


## LLM-as-judge: System 1 only

Judges `system1_report.md` with the custom System 1 rubric `rubric_s1_v2.yaml`
(0-10 scale in 0.5 steps, evaluation principles, strengths/weaknesses/rationale/confidence
output). One API call; saves the prompt, the raw response, and `eval_seed_101_s1_v2_<ts>.json`
under `evaluation_trial/trial1/s1_evaluation/`. To iterate on the rubric, edit
`rubric_s1_v2.yaml`; to iterate on the evaluator persona, edit the Evaluator prompts cell.

In [10]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# repo root = the directory that contains evaluation_refined/
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "evaluation_refined").is_dir()
)
for p in (PROJECT_ROOT / "evaluation_refined", PROJECT_ROOT / "scripts", PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import run_refined_evaluation as rre

from openai import OpenAI

# ============================================================
# INPUTS
# ============================================================

S1_REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system1_report.md"
)
# v2 rubric: 0-10 scale in 0.5 steps, evaluation principles, and a
# strengths/weaknesses/rationale/confidence output schema.
S1_RUBRIC_PATH = PROJECT_ROOT / "evaluation_refined/rubric_s1_v2.yaml"
S1_JUDGE_MODEL = "gpt-5.6-sol"
S1_RUN_LABEL = "seed_101_s1_v2"
S1_OUT_DIR = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/s1_evaluation"
)
S1_API_KEY_FILE = Path.home() / "Github/api_key.env"

# ============================================================
# Build the prompts from the v2 rubric
# ============================================================

s1_rubric = rre.load_rubric(S1_RUBRIC_PATH)
S1_DIM_KEYS = list(s1_rubric["dimensions"])
# Guard against a half-parsed rubric (a mis-indented block silently
# drops dimensions or leaks their fields into `dimensions`).
assert S1_DIM_KEYS, "rubric has no dimensions"
for _dk in S1_DIM_KEYS:
    _d = s1_rubric["dimensions"][_dk]
    assert _dk.startswith("system1_") and isinstance(_d, dict) \
        and {"rubric", "short_name", "name", "weight"} <= set(_d), (
        f"rubric dimension {_dk!r} malformed (check YAML indentation)"
    )
s1_model = S1_JUDGE_MODEL or s1_rubric.get("judge_model", "gpt-4.1")
s1_smin = float(s1_rubric.get("score_min", 0))
s1_smax = float(s1_rubric.get("score_max", 10))
s1_step = float(s1_rubric.get("score_step", 0.5))

report_text = S1_REPORT_MD.read_text(encoding="utf-8")
query_line = next(
    (ln.strip() for ln in report_text.splitlines() if ln.startswith("Find ")),
    "",
)

# Edit the prompt in the 'Evaluator prompts' cell above.
s1_system_prompt = S1_JUDGE_SYSTEM_PROMPT

scale_block = "\n".join(
    f"- {line}" for line in (s1_rubric.get("ordinal_scale_lines") or [])
)
rubric_text = f"### Score scale ({s1_smin:g}-{s1_smax:g}, step {s1_step:g})\n{scale_block}\n"
if s1_rubric.get("evaluation_principles"):
    rubric_text += f"\n### Evaluation principles\n{s1_rubric['evaluation_principles']}\n"
for dk in S1_DIM_KEYS:
    dim = s1_rubric["dimensions"][dk]
    rubric_text += f"\n### {dim['name']} (weight: {dim['weight']})\n"
    if dim.get("primary_question"):
        rubric_text += f"Primary question: {dim['primary_question'].strip()}\n\n"
    rubric_text += f"{dim['rubric']}\n"

s1_output_block = ""
if s1_rubric.get("output_instructions"):
    s1_output_block += s1_rubric["output_instructions"].strip() + "\n\n"
s1_output_block += (
    "Respond with ONLY a JSON object in this exact structure:\n\n"
    f"```\n{s1_rubric.get('output_schema', '').strip()}\n```"
)

s1_user_prompt = (
    f"## Material Substitution Query\n\n{query_line}\n\n"
    f"## Evaluation Rubric (System 1 only)\n{rubric_text}\n\n"
    f"## System 1 Report\n\n{report_text}\n\n"
    f"## Required Output Format\n\n{s1_output_block}"
)

print(
    f"judge model: {s1_model} | rubric: {S1_RUBRIC_PATH.name} | "
    f"prompt: {len(s1_user_prompt):,} chars (system {len(s1_system_prompt):,})"
)

# ============================================================
# Run the judge (one API call)
# ============================================================

s1_api_key = os.environ.get("OPENAI_API_KEY")
if not s1_api_key and S1_API_KEY_FILE.exists():
    for line in open(S1_API_KEY_FILE):
        if line.strip().startswith("OPENAI_API_KEY="):
            s1_api_key = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
assert s1_api_key, "no API key: set OPENAI_API_KEY or S1_API_KEY_FILE"

S1_OUT_DIR.mkdir(parents=True, exist_ok=True)
s1_ts = datetime.utcnow().strftime("%Y%m%d%H%M")
(S1_OUT_DIR / f"judge_prompt_{s1_ts}.txt").write_text(
    "### SYSTEM PROMPT\n" + s1_system_prompt
    + "\n\n### USER PROMPT\n" + s1_user_prompt,
    encoding="utf-8",
)

t0 = time.time()
s1_parsed, s1_meta, s1_deviations = rre.call_judge(
    OpenAI(api_key=s1_api_key), s1_model,
    s1_system_prompt, s1_user_prompt, s1_rubric,
)
s1_elapsed = time.time() - t0
print(
    f"judge responded in {s1_elapsed:.1f}s | "
    f"finish_reason={s1_meta.get('finish_reason')} | "
    f"usage={s1_meta.get('usage')}"
)
if s1_deviations:
    print("protocol deviations:", "; ".join(s1_deviations))
assert "error" not in s1_parsed, s1_parsed
(S1_OUT_DIR / f"judge_raw_{s1_ts}.json").write_text(
    json.dumps(
        {"response": s1_meta.pop("raw_text", ""), "meta": s1_meta},
        indent=2, ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ============================================================
# Scores, comments, result JSON
# ============================================================

# v2 schema puts dimension keys at the top level; tolerate an "A" wrapper.
s1_block = s1_parsed.get("A") if isinstance(s1_parsed.get("A"), dict) else s1_parsed


def _snap_score(value):
    """Clamp to the rubric range and snap to the 0.5 grid."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    v = max(s1_smin, min(s1_smax, v))
    return round(v / s1_step) * s1_step


s1_scores = {}
for dk in S1_DIM_KEYS:
    entry = s1_block.get(dk) or {}
    s1_scores[dk] = {
        "score": _snap_score(entry.get("score")),
        "strengths": entry.get("strengths", []),
        "weaknesses": entry.get("weaknesses", []),
        "rationale": entry.get("rationale", ""),
        "confidence": entry.get("confidence", ""),
    }

valid = [s1_scores[dk]["score"] for dk in S1_DIM_KEYS
         if s1_scores[dk]["score"] is not None]
s1_mean = round(sum(valid) / len(valid), 2) if valid else None

hdr = [s1_rubric["dimensions"][dk]["short_name"] for dk in S1_DIM_KEYS]
print(f"\n{'run':<18}" + "".join(f"{h:<10}" for h in hdr) + f"{'Mean':>7}")
print(
    f"{S1_RUN_LABEL:<18}"
    + "".join(f"{str(s1_scores[dk]['score']):<10}" for dk in S1_DIM_KEYS)
    + f"{s1_mean:>7}"
)
print()
for dk in S1_DIM_KEYS:
    s = s1_scores[dk]
    print(f"[{s1_rubric['dimensions'][dk]['name']}]  "
          f"score = {s['score']} (confidence: {s['confidence']})")
    for tag, items in (("+", s["strengths"]), ("-", s["weaknesses"])):
        for item in items:
            print(f"   {tag} {item}")
    print(f"   rationale: {s['rationale']}\n")

s1_result = {
    "study": "System 1 only: LLM judge on system1_report.md (rubric v2, 0-10 scale)",
    "query_name": "Query1",
    "query_sentence": query_line,
    "input_format": "system1_report.md (query, initial RAG, Q&A with sources, derived outputs)",
    "report_path": str(S1_REPORT_MD),
    "rubric_path": str(S1_RUBRIC_PATH),
    "dimensions": S1_DIM_KEYS,
    "scores": {S1_RUN_LABEL: s1_scores},
    "mean_score": s1_mean,
    "judge_model": s1_model,
    "protocol_deviation": "; ".join(s1_deviations) if s1_deviations else None,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "judge_elapsed_seconds": round(s1_elapsed, 1),
    "judge_completion": s1_meta,
}
s1_outfile = S1_OUT_DIR / f"eval_{S1_RUN_LABEL}_{s1_ts}.json"
s1_outfile.write_text(
    json.dumps(s1_result, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("saved:", s1_outfile)

judge model: gpt-5.6-sol | rubric: rubric_s1_v2.yaml | prompt: 37,384 chars (system 952)
judge responded in 41.6s | finish_reason=stop | usage={'prompt_tokens': 8746, 'completion_tokens': 2135, 'total_tokens': 10881}
protocol deviations: max_tokens renamed to max_completion_tokens (model requirement); temperature omitted: model rejected temperature=0 (Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the d)

run               S1-Compl  S1-Rel    S1-Sci    S1-Clar      Mean
seed_101_s1_v2    8.0       7.5       7.0       7.5           7.5

[System 1 — Completeness]  score = 8.0 (confidence: high)
   + The final set covers all user-prioritized categories: acids, bases, protic/aprotic solvents, oils/fuels, salts, flexibility, transparency, and gas permeability.
   + It also includes useful secondary categories for thermal behavior, extrusion/molding, PFAS composition, contamination control, regulatory compliance, cost, and a

## LLM-as-judge: System 2 only

Judges `system2_report.md` with the System 2 rubric `rubric_s2_v2.yaml` (0-10 scale in 0.5
steps, evaluation principles, strengths/weaknesses/rationale/confidence output) on four
dimensions: requirement alignment, search-space exploration, evidence and validation
quality, adaptive reasoning. One API
call; saves the prompt, the raw response, and `eval_seed_101_s2_v2_<ts>.json` under
`evaluation_trial/trial1/s2_evaluation/`. Edit `rubric_s2_v2.yaml` to change the metrics.

In [11]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# repo root = the directory that contains evaluation_refined/
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "evaluation_refined").is_dir()
)
for p in (PROJECT_ROOT / "evaluation_refined", PROJECT_ROOT / "scripts", PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import run_refined_evaluation as rre

from openai import OpenAI

# ============================================================
# INPUTS
# ============================================================

S2_REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system2_report.md"
)
# v2 rubric: 0-10 scale in 0.5 steps, evaluation principles, and a
# strengths/weaknesses/rationale/confidence output schema.
S2_RUBRIC_PATH = PROJECT_ROOT / "evaluation_refined/rubric_s2_v2.yaml"
S2_JUDGE_MODEL = "gpt-5.6-sol"
S2_RUN_LABEL = "seed_101_s2_v2"
S2_OUT_DIR = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/s2_evaluation"
)
S2_API_KEY_FILE = Path.home() / "Github/api_key.env"

# ============================================================
# Build the prompts from the v2 rubric
# ============================================================

s2_rubric = rre.load_rubric(S2_RUBRIC_PATH)
S2_DIM_KEYS = list(s2_rubric["dimensions"])
# Guard against a half-parsed rubric (a mis-indented block silently
# drops dimensions or leaks their fields into `dimensions`).
assert S2_DIM_KEYS, "rubric has no dimensions"
for _dk in S2_DIM_KEYS:
    _d = s2_rubric["dimensions"][_dk]
    assert _dk.startswith("system2_") and isinstance(_d, dict) \
        and {"rubric", "short_name", "name", "weight"} <= set(_d), (
        f"rubric dimension {_dk!r} malformed (check YAML indentation)"
    )
s2_model = S2_JUDGE_MODEL or s2_rubric.get("judge_model", "gpt-4.1")
s2_smin = float(s2_rubric.get("score_min", 0))
s2_smax = float(s2_rubric.get("score_max", 10))
s2_step = float(s2_rubric.get("score_step", 0.5))

report_text = S2_REPORT_MD.read_text(encoding="utf-8")
query_line = (
    "Find a substitute material for PFAS material THV (elastomeric terpolymer "
    "of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) "
    "that has broad chemical resistance, including acids, bases, protic and "
    "aprotic solvents, oils, and salt solutions. Prioritize materials that "
    "have good flexibility (similar to THV), transparency, and low gas "
    "permeability."
)

# Edit the prompt in the 'Evaluator prompts' cell above.
s2_system_prompt = S2_JUDGE_SYSTEM_PROMPT

scale_block = "\n".join(
    f"- {line}" for line in (s2_rubric.get("ordinal_scale_lines") or [])
)
rubric_text = f"### Score scale ({s2_smin:g}-{s2_smax:g}, step {s2_step:g})\n{scale_block}\n"
if s2_rubric.get("evaluation_principles"):
    rubric_text += f"\n### Evaluation principles\n{s2_rubric['evaluation_principles']}\n"
for dk in S2_DIM_KEYS:
    dim = s2_rubric["dimensions"][dk]
    rubric_text += f"\n### {dim['name']} (weight: {dim['weight']})\n"
    if dim.get("primary_question"):
        rubric_text += f"Primary question: {dim['primary_question'].strip()}\n\n"
    rubric_text += f"{dim['rubric']}\n"

s2_output_block = ""
if s2_rubric.get("output_instructions"):
    s2_output_block += s2_rubric["output_instructions"].strip() + "\n\n"
s2_output_block += (
    "Respond with ONLY a JSON object in this exact structure:\n\n"
    f"```\n{s2_rubric.get('output_schema', '').strip()}\n```"
)

s2_user_prompt = (
    f"## Material Substitution Query\n\n{query_line}\n\n"
    f"## Evaluation Rubric (System 2 only)\n{rubric_text}\n\n"
    f"## System 2 Report\n\n{report_text}\n\n"
    f"## Required Output Format\n\n{s2_output_block}"
)

print(
    f"judge model: {s2_model} | rubric: {S2_RUBRIC_PATH.name} | "
    f"prompt: {len(s2_user_prompt):,} chars (system {len(s2_system_prompt):,})"
)

# ============================================================
# Run the judge (one API call)
# ============================================================

s2_api_key = os.environ.get("OPENAI_API_KEY")
if not s2_api_key and S2_API_KEY_FILE.exists():
    for line in open(S2_API_KEY_FILE):
        if line.strip().startswith("OPENAI_API_KEY="):
            s2_api_key = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
assert s2_api_key, "no API key: set OPENAI_API_KEY or S2_API_KEY_FILE"

S2_OUT_DIR.mkdir(parents=True, exist_ok=True)
s2_ts = datetime.utcnow().strftime("%Y%m%d%H%M")
(S2_OUT_DIR / f"judge_prompt_{s2_ts}.txt").write_text(
    "### SYSTEM PROMPT\n" + s2_system_prompt
    + "\n\n### USER PROMPT\n" + s2_user_prompt,
    encoding="utf-8",
)

t0 = time.time()
s2_parsed, s2_meta, s2_deviations = rre.call_judge(
    OpenAI(api_key=s2_api_key), s2_model,
    s2_system_prompt, s2_user_prompt, s2_rubric,
)
s2_elapsed = time.time() - t0
print(
    f"judge responded in {s2_elapsed:.1f}s | "
    f"finish_reason={s2_meta.get('finish_reason')} | "
    f"usage={s2_meta.get('usage')}"
)
if s2_deviations:
    print("protocol deviations:", "; ".join(s2_deviations))
assert "error" not in s2_parsed, s2_parsed
(S2_OUT_DIR / f"judge_raw_{s2_ts}.json").write_text(
    json.dumps(
        {"response": s2_meta.pop("raw_text", ""), "meta": s2_meta},
        indent=2, ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ============================================================
# Scores, comments, result JSON
# ============================================================

# v2 schema puts dimension keys at the top level; tolerate an "A" wrapper.
s2_block = s2_parsed.get("A") if isinstance(s2_parsed.get("A"), dict) else s2_parsed


def _snap_score(value):
    """Clamp to the rubric range and snap to the 0.5 grid."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    v = max(s2_smin, min(s2_smax, v))
    return round(v / s2_step) * s2_step


s2_scores = {}
for dk in S2_DIM_KEYS:
    entry = s2_block.get(dk) or {}
    s2_scores[dk] = {
        "score": _snap_score(entry.get("score")),
        "strengths": entry.get("strengths", []),
        "weaknesses": entry.get("weaknesses", []),
        "rationale": entry.get("rationale", ""),
        "confidence": entry.get("confidence", ""),
    }

valid = [s2_scores[dk]["score"] for dk in S2_DIM_KEYS
         if s2_scores[dk]["score"] is not None]
s2_mean = round(sum(valid) / len(valid), 2) if valid else None

hdr = [s2_rubric["dimensions"][dk]["short_name"] for dk in S2_DIM_KEYS]
print(f"\n{'run':<18}" + "".join(f"{h:<10}" for h in hdr) + f"{'Mean':>7}")
print(
    f"{S2_RUN_LABEL:<18}"
    + "".join(f"{str(s2_scores[dk]['score']):<10}" for dk in S2_DIM_KEYS)
    + f"{s2_mean:>7}"
)
print()
for dk in S2_DIM_KEYS:
    s = s2_scores[dk]
    print(f"[{s2_rubric['dimensions'][dk]['name']}]  "
          f"score = {s['score']} (confidence: {s['confidence']})")
    for tag, items in (("+", s["strengths"]), ("-", s["weaknesses"])):
        for item in items:
            print(f"   {tag} {item}")
    print(f"   rationale: {s['rationale']}\n")

s2_result = {
    "study": "System 1 only: LLM judge on system2_report.md (rubric v2, 0-10 scale)",
    "query_name": "Query1",
    "query_sentence": query_line,
    "input_format": "system2_report.md (query, initial RAG, Q&A with sources, derived outputs)",
    "report_path": str(S2_REPORT_MD),
    "rubric_path": str(S2_RUBRIC_PATH),
    "dimensions": S2_DIM_KEYS,
    "scores": {S2_RUN_LABEL: s2_scores},
    "mean_score": s2_mean,
    "judge_model": s2_model,
    "protocol_deviation": "; ".join(s2_deviations) if s2_deviations else None,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "judge_elapsed_seconds": round(s2_elapsed, 1),
    "judge_completion": s2_meta,
}
s2_outfile = S2_OUT_DIR / f"eval_{S2_RUN_LABEL}_{s2_ts}.json"
s2_outfile.write_text(
    json.dumps(s2_result, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("saved:", s2_outfile)

judge model: gpt-5.6-sol | rubric: rubric_s2_v2.yaml | prompt: 95,295 chars (system 1,387)
judge responded in 29.3s | finish_reason=stop | usage={'prompt_tokens': 25195, 'completion_tokens': 1592, 'total_tokens': 26787}
protocol deviations: max_tokens renamed to max_completion_tokens (model requirement); temperature omitted: model rejected temperature=0 (Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the d)

run               S2-Algn   S2-Expl   S2-Evid   S2-Adapt     Mean
seed_101_s2_v2    8.0       7.5       7.5       7.0           7.5

[System 2 — Requirement alignment]  score = 8.0 (confidence: high)
   + Major requirements repeatedly drive validation: Iterations 2–4 ask directly about PFAS certification, chemical immersion, flexibility, transparency, gas permeability, and high-temperature aging.
   + Rejections are explicitly tied to hard constraints: Iteration 1 is rejected for fluorinated FKM, while Iterations 2

## LLM-as-judge: System 3 only

Judges `system3_report.md` with the System 3 rubric `rubric_s3_v2.yaml` (0-10 scale in 0.5
steps, evaluation principles, strengths/weaknesses/rationale/confidence output). One API
call; saves the prompt, the raw response, and `eval_seed_101_s3_v2_<ts>.json` under
`evaluation_trial/trial1/s3_evaluation/`. Edit `rubric_s3_v2.yaml` to change the metrics.

In [19]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# repo root = the directory that contains evaluation_refined/
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "evaluation_refined").is_dir()
)
for p in (PROJECT_ROOT / "evaluation_refined", PROJECT_ROOT / "scripts", PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import run_refined_evaluation as rre

from openai import OpenAI

# ============================================================
# INPUTS
# ============================================================

S3_REPORT_MD = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/system3_report.md"
)
# v2 rubric: 0-10 scale in 0.5 steps, evaluation principles, and a
# strengths/weaknesses/rationale/confidence output schema.
S3_RUBRIC_PATH = PROJECT_ROOT / "evaluation_refined/rubric_s3_v2.yaml"
S3_JUDGE_MODEL = "gpt-5.6-sol"
S3_RUN_LABEL = "seed_101_s3_v2"
S3_OUT_DIR = Path(
    "/home/palsubhadeep/Github/MARS/evaluation_trial/trial1/s3_evaluation"
)
S3_API_KEY_FILE = Path.home() / "Github/api_key.env"

# ============================================================
# Build the prompts from the v2 rubric
# ============================================================

s3_rubric = rre.load_rubric(S3_RUBRIC_PATH)
S3_DIM_KEYS = list(s3_rubric["dimensions"])
# Guard against a half-parsed rubric (a mis-indented block silently
# drops dimensions or leaks their fields into `dimensions`).
assert S3_DIM_KEYS, "rubric has no dimensions"
for _dk in S3_DIM_KEYS:
    _d = s3_rubric["dimensions"][_dk]
    assert _dk.startswith("system3_") and isinstance(_d, dict) \
        and {"rubric", "short_name", "name", "weight"} <= set(_d), (
        f"rubric dimension {_dk!r} malformed (check YAML indentation)"
    )
s3_model = S3_JUDGE_MODEL or s3_rubric.get("judge_model", "gpt-4.1")
s3_smin = float(s3_rubric.get("score_min", 0))
s3_smax = float(s3_rubric.get("score_max", 10))
s3_step = float(s3_rubric.get("score_step", 0.5))

report_text = S3_REPORT_MD.read_text(encoding="utf-8")
query_line = (
    "Find a substitute material for PFAS material THV (elastomeric terpolymer "
    "of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) "
    "that has broad chemical resistance, including acids, bases, protic and "
    "aprotic solvents, oils, and salt solutions. Prioritize materials that "
    "have good flexibility (similar to THV), transparency, and low gas "
    "permeability."
)

# Edit the prompt in the 'Evaluator prompts' cell above.
s3_system_prompt = S3_JUDGE_SYSTEM_PROMPT

scale_block = "\n".join(
    f"- {line}" for line in (s3_rubric.get("ordinal_scale_lines") or [])
)
rubric_text = f"### Score scale ({s3_smin:g}-{s3_smax:g}, step {s3_step:g})\n{scale_block}\n"
if s3_rubric.get("evaluation_principles"):
    rubric_text += f"\n### Evaluation principles\n{s3_rubric['evaluation_principles']}\n"
for dk in S3_DIM_KEYS:
    dim = s3_rubric["dimensions"][dk]
    rubric_text += f"\n### {dim['name']} (weight: {dim['weight']})\n"
    if dim.get("primary_question"):
        rubric_text += f"Primary question: {dim['primary_question'].strip()}\n\n"
    rubric_text += f"{dim['rubric']}\n"

s3_output_block = ""
if s3_rubric.get("output_instructions"):
    s3_output_block += s3_rubric["output_instructions"].strip() + "\n\n"
s3_output_block += (
    "Respond with ONLY a JSON object in this exact structure:\n\n"
    f"```\n{s3_rubric.get('output_schema', '').strip()}\n```"
)

s3_user_prompt = (
    f"## Material Substitution Query\n\n{query_line}\n\n"
    f"## Evaluation Rubric (System 3 only)\n{rubric_text}\n\n"
    f"## System 3 Report\n\n{report_text}\n\n"
    f"## Required Output Format\n\n{s3_output_block}"
)

print(
    f"judge model: {s3_model} | rubric: {S3_RUBRIC_PATH.name} | "
    f"prompt: {len(s3_user_prompt):,} chars (system {len(s3_system_prompt):,})"
)

# ============================================================
# Run the judge (one API call)
# ============================================================

s3_api_key = os.environ.get("OPENAI_API_KEY")
if not s3_api_key and S3_API_KEY_FILE.exists():
    for line in open(S3_API_KEY_FILE):
        if line.strip().startswith("OPENAI_API_KEY="):
            s3_api_key = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
assert s3_api_key, "no API key: set OPENAI_API_KEY or S3_API_KEY_FILE"

S3_OUT_DIR.mkdir(parents=True, exist_ok=True)
s3_ts = datetime.utcnow().strftime("%Y%m%d%H%M")
(S3_OUT_DIR / f"judge_prompt_{s3_ts}.txt").write_text(
    "### SYSTEM PROMPT\n" + s3_system_prompt
    + "\n\n### USER PROMPT\n" + s3_user_prompt,
    encoding="utf-8",
)

t0 = time.time()
s3_parsed, s3_meta, s3_deviations = rre.call_judge(
    OpenAI(api_key=s3_api_key), s3_model,
    s3_system_prompt, s3_user_prompt, s3_rubric,
)
s3_elapsed = time.time() - t0
print(
    f"judge responded in {s3_elapsed:.1f}s | "
    f"finish_reason={s3_meta.get('finish_reason')} | "
    f"usage={s3_meta.get('usage')}"
)
if s3_deviations:
    print("protocol deviations:", "; ".join(s3_deviations))
assert "error" not in s3_parsed, s3_parsed
(S3_OUT_DIR / f"judge_raw_{s3_ts}.json").write_text(
    json.dumps(
        {"response": s3_meta.pop("raw_text", ""), "meta": s3_meta},
        indent=2, ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ============================================================
# Scores, comments, result JSON
# ============================================================

# v2 schema puts dimension keys at the top level; tolerate an "A" wrapper.
s3_block = s3_parsed.get("A") if isinstance(s3_parsed.get("A"), dict) else s3_parsed


def _snap_score(value):
    """Clamp to the rubric range and snap to the 0.5 grid."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    v = max(s3_smin, min(s3_smax, v))
    return round(v / s3_step) * s3_step


s3_scores = {}
for dk in S3_DIM_KEYS:
    entry = s3_block.get(dk) or {}
    s3_scores[dk] = {
        "score": _snap_score(entry.get("score")),
        "strengths": entry.get("strengths", []),
        "weaknesses": entry.get("weaknesses", []),
        "rationale": entry.get("rationale", ""),
        "confidence": entry.get("confidence", ""),
    }

valid = [s3_scores[dk]["score"] for dk in S3_DIM_KEYS
         if s3_scores[dk]["score"] is not None]
s3_mean = round(sum(valid) / len(valid), 2) if valid else None

hdr = [s3_rubric["dimensions"][dk]["short_name"] for dk in S3_DIM_KEYS]
print(f"\n{'run':<18}" + "".join(f"{h:<10}" for h in hdr) + f"{'Mean':>7}")
print(
    f"{S3_RUN_LABEL:<18}"
    + "".join(f"{str(s3_scores[dk]['score']):<10}" for dk in S3_DIM_KEYS)
    + f"{s3_mean:>7}"
)
print()
for dk in S3_DIM_KEYS:
    s = s3_scores[dk]
    print(f"[{s3_rubric['dimensions'][dk]['name']}]  "
          f"score = {s['score']} (confidence: {s['confidence']})")
    for tag, items in (("+", s["strengths"]), ("-", s["weaknesses"])):
        for item in items:
            print(f"   {tag} {item}")
    print(f"   rationale: {s['rationale']}\n")

s3_result = {
    "study": "System 1 only: LLM judge on system3_report.md (rubric v2, 0-10 scale)",
    "query_name": "Query1",
    "query_sentence": query_line,
    "input_format": "system3_report.md (query, initial RAG, Q&A with sources, derived outputs)",
    "report_path": str(S3_REPORT_MD),
    "rubric_path": str(S3_RUBRIC_PATH),
    "dimensions": S3_DIM_KEYS,
    "scores": {S3_RUN_LABEL: s3_scores},
    "mean_score": s3_mean,
    "judge_model": s3_model,
    "protocol_deviation": "; ".join(s3_deviations) if s3_deviations else None,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "judge_elapsed_seconds": round(s3_elapsed, 1),
    "judge_completion": s3_meta,
}
s3_outfile = S3_OUT_DIR / f"eval_{S3_RUN_LABEL}_{s3_ts}.json"
s3_outfile.write_text(
    json.dumps(s3_result, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("saved:", s3_outfile)

judge model: gpt-5.6-sol | rubric: rubric_s3_v2.yaml | prompt: 55,442 chars (system 1,262)
judge responded in 40.7s | finish_reason=stop | usage={'prompt_tokens': 16487, 'completion_tokens': 2126, 'total_tokens': 18613}
protocol deviations: max_tokens renamed to max_completion_tokens (model requirement); temperature omitted: model rejected temperature=0 (Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the d)

run               S3-Route  S3-Prac   S3-Comp   S3-Ind       Mean
seed_101_s3_v2    7.5       7.0       7.0       7.0          7.12

[System 3 — Manufacturing route coherence]  score = 7.5 (confidence: high)
   + The recipe provides a recognizable end-to-end route: receipt and inspection (Step 1), drying (Step 2), TiO2 compounding (Step 3), extrusion (Step 4), cooling (Step 5), final shaping (Step 6), QC (Step 9), and packaging/storage (Step 10).
   + Material flow is generally sensible, and the route accommodates 

## Ablation: 3-agent (no RAG / no KG), seed 101

Builds three reports from `results_seedstudy/seed_101/Query1/ablation_3agent.json` that mirror the seed-study report sections; sections the condition cannot produce are kept and marked *not produced* so the judge scores absence knowingly.

In [ ]:
from pathlib import Path
import json

# ============================================================
# 3-agent ablations, seed 101, Query1: "3agent" (no RAG / no KG)
# and "3agent_rag" (same three prompts + the identical pre-retrieved
# packet, fetched by the harness with the raw query, pasted into every
# stage prompt). Renders three markdown reports per condition that
# mirror the seed-study system reports section-for-section; sections
# the condition cannot produce are kept and marked as not produced.
# ============================================================

ABL3_RUN_DIR = Path("/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1")
ABL3_CONDITIONS = ["3agent", "3agent_rag"]
ABL3_OUT_DIR = Path("/home/palsubhadeep/Github/MARS/evaluation_trial/ablations")

NOT_PRODUCED = (
    "*Not produced in this condition (no RAG, no KG by design). "
    "This section exists in the full MARS pipeline reports.*"
)


def _fmt_recipe_step(step):
    if not isinstance(step, dict):
        return [f"- {step}"]
    idx = step.get("step_index", "?")
    name = step.get("process_step") or step.get("step") or step.get("name") or ""
    lines = [f"**Step {idx}. {name}**"]
    for key, val in step.items():
        if key in ("step_index", "process_step", "step", "name"):
            continue
        if isinstance(val, dict):
            lines.append(f"- {key}:")
            lines += [f"  - {k}: {v}" for k, v in val.items()]
        elif isinstance(val, list):
            lines.append(f"- {key}: " + "; ".join(str(v) for v in val))
        else:
            lines.append(f"- {key}: {val}")
    return lines


def build_3agent_reports(condition, run_dir, out_dir):
    with open(run_dir / f"ablation_{condition}.json", encoding="utf-8") as f:
        abl = json.load(f)

    meta = abl.get("metadata", {})
    query = abl.get("query", {})
    props = (abl.get("required_material_properties") or {}).get("properties", [])
    cons = (abl.get("required_material_properties") or {}).get("constraints", [])
    cand = (abl.get("candidate_selection") or {}).get("final_candidate") or {}
    rejected = (abl.get("candidate_selection") or {}).get("rejected_candidates", [])
    mfg = abl.get("manufacturing_process") or {}
    raw = abl.get("raw_responses") or {}
    context = raw.get("retrieved_context")
    manifest = raw.get("retrieval_manifest") or {}
    has_context = bool(context)

    grounding_clause = (
        "with an identical pre-retrieved RAG+KG evidence packet pasted into "
        "every stage's prompt (retrieved once by the harness with the raw "
        "query; no agent directed any retrieval)"
        if has_context else
        "**no RAG retrieval and no knowledge graph**"
    )
    condition_note = (
        f"> **Condition:** `{meta.get('ablation_condition')}` ablation, three sequential "
        f"LLM calls (property extraction, candidate selection, manufacturability), "
        f"{grounding_clause}. "
        f"Run `{meta.get('pipeline_run_id')}`, total duration "
        f"{meta.get('duration_seconds', 0):.0f}s across all three calls. "
        f"Sections below marked *not produced* are absent because this condition "
        f"cannot generate them, not because they were removed from the report."
    )

    evidence_section = []
    if has_context:
        rag_man = manifest.get("rag") or {}
        kg_man = manifest.get("kg") or {}
        evidence_section = [
            "## Pre-retrieved evidence packet (identical in all three stage prompts)",
            "",
            "Fetched once by the harness with the raw query before any agent ran; "
            "no agent chose queries or saw stage-specific evidence.",
            "",
        ]
        for src, ids in rag_man.items():
            shown = ", ".join(str(i) for i in ids) if ids else "(retrieval failed)"
            evidence_section.append(f"- RAG [{src}]: {len(ids or [])} documents: {shown}")
        for kg, info in kg_man.items():
            if info:
                evidence_section.append(
                    f"- KG [{kg}]: {info.get('matched_nodes')} matched nodes, "
                    f"{info.get('found_paths')} paths"
                )
            else:
                evidence_section.append(f"- KG [{kg}]: search failed")
        if manifest.get("material_db_entries"):
            evidence_section.append(
                f"- Material database listing: {manifest['material_db_entries']} entries"
            )
        evidence_section += [
            "",
            "### Full packet text (verbatim)",
            "",
            "```",
            (context or "").strip(),
            "```",
        ]

    # ---------------- System 1 analog: agent 1 ----------------
    s1 = [
        f"# System 1 report ({condition} ablation): requirement extraction",
        "",
        condition_note,
        "",
        "## 1. The query",
        "",
        query.get("sentence", ""),
        "",
        f"- Material X: {query.get('material_X', '')}",
        f"- Application Y: {query.get('application_Y', '')}",
        "",
        "## 2. Initial RAG retrieval",
        "",
        ("*See the shared evidence packet section at the end; retrieval used the "
         "raw query once, fetched by the harness, not by any agent.*"
         if has_context else NOT_PRODUCED),
        "",
        "## 3. Generated questions by LLM",
        "",
        NOT_PRODUCED,
        "",
        "*Instead, this condition made a single LLM call. Its full response, "
        "the only provenance that exists for section 4, is:*",
        "",
        "### 3.1 Agent response (single call, verbatim)",
        "",
        "```",
        (raw.get("agent1_properties") or "").strip(),
        "```",
        "",
        "## 4. Derived outputs by LLM",
        "",
        "### 4.1 Derived material properties",
        "",
    ]
    s1 += [f"{i}. {p}" for i, p in enumerate(props, 1)] or ["(none)"]
    s1 += ["", "### 4.2 Derived hard constraints", ""]
    s1 += [f"{i}. {c}" for i, c in enumerate(cons, 1)] or ["(none)"]

    # ---------------- System 2 analog: agent 2 ----------------
    s2 = [
        f"# System 2 report ({condition} ablation): candidate selection",
        "",
        condition_note,
        "",
        "## 1. The extracted properties and hard constraints from System 1",
        "",
        "### Properties",
        "",
    ]
    s2 += [f"{i}. {p}" for i, p in enumerate(props, 1)] or ["(none)"]
    s2 += ["", "### Hard constraints", ""]
    s2 += [f"{i}. {c}" for i, c in enumerate(cons, 1)] or ["(none)"]
    s2 += [
        "",
        "## 2. KG call to nodes related to material properties",
        "",
        NOT_PRODUCED,
        "",
        "## 3. Path-node classification (material or not)",
        "",
        NOT_PRODUCED,
        "",
        "## 4. Whole-subgraph classification",
        "",
        NOT_PRODUCED,
        "",
        "## 5. Candidate material proposal and verification",
        "",
        f"*This condition proposes exactly once; there is no PFAS check, no validation "
        f"questions, no retrieved evidence, no feasibility verdict, and no rejection "
        f"loop. Rejected candidates recorded: {len(rejected)}.*",
        "",
        "### 5.1 Proposed candidate",
        "",
        f"- Material name: {cand.get('material_name', '')}",
        f"- Material class: {cand.get('material_class', '')}",
        "",
        "### 5.2 Justification (verbatim)",
        "",
        cand.get("justification", "") or "(none)",
        "",
        "### 5.3 PFAS compliance check",
        "",
        NOT_PRODUCED,
        "",
        "### 5.4 Validation questions and retrieved-evidence answers",
        "",
        NOT_PRODUCED,
        "",
        "### 5.5 Feasibility verdict",
        "",
        NOT_PRODUCED,
        "",
        "### 5.6 Agent response (single call, verbatim)",
        "",
        "```",
        (raw.get("agent2_candidate") or "").strip(),
        "```",
    ]

    # ---------------- System 3 analog: agent 3 ----------------
    recipe = mfg.get("process_recipe") or []
    s3 = [
        f"# System 3 report ({condition} ablation): manufacturability",
        "",
        condition_note,
        "",
        "## 1. Proposed candidate and justification from System 2",
        "",
        f"- Candidate material: {cand.get('material_name', '')}",
        f"- Material class: {cand.get('material_class', '')}",
        "",
        "Justification handed over:",
        "",
        cand.get("justification", "") or "(none)",
        "",
        "## 2. Single vs composite decomposition",
        "",
        NOT_PRODUCED,
        "",
        "## 3. Manufacturing process queries with retrieved evidence",
        "",
        NOT_PRODUCED,
        "",
        "## 4. Feasibility questions and evidence-based answers",
        "",
        NOT_PRODUCED,
        "",
        "## 5. Feasibility verdict",
        "",
        f"- Status: **{mfg.get('status', 'unknown')}**",
        f"- Blocking constraints: {mfg.get('blocking_constraints') or '(none listed)'}",
        f"- Feedback to System 2: {mfg.get('feedback_to_system2') or '(none)'}",
        "",
        "*No verdict reasoning, confidence, or supporting evidence exists in this "
        "condition; the status above is the entire verdict.*",
        "",
        f"## 6. Process recipe ({len(recipe)} steps)",
        "",
    ]
    for step in recipe:
        s3 += _fmt_recipe_step(step)
        s3.append("")
    s3 += [
        "### Backing evidence",
        "",
        NOT_PRODUCED,
        "",
        "### Agent response (single call, verbatim)",
        "",
        "```",
        (raw.get("agent3_manufacturing") or "").strip(),
        "```",
    ]

    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for n, lines in ((1, s1), (2, s2), (3, s3)):
        if evidence_section:
            lines = lines + [""] + evidence_section
        path = out_dir / f"{condition}_system{n}_report.md"
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        paths[n] = path
    return paths, abl


for _cond in ABL3_CONDITIONS:
    abl3_paths, abl3_data = build_3agent_reports(_cond, ABL3_RUN_DIR, ABL3_OUT_DIR)
    _m = abl3_data["metadata"]
    print(f"condition: {_m.get('ablation_condition')} | run {_m.get('pipeline_run_id')} | "
          f"{_m.get('duration_seconds', 0):.0f}s")
    print(f"  properties: {len(abl3_data['required_material_properties']['properties'])} | "
          f"constraints: {len(abl3_data['required_material_properties']['constraints'])}")
    print(f"  candidate: {(abl3_data['candidate_selection']['final_candidate'] or {}).get('material_name')}")
    print(f"  mfg status: {abl3_data['manufacturing_process'].get('status')} | "
          f"recipe steps: {len(abl3_data['manufacturing_process'].get('process_recipe') or [])}")
    for n, rp in abl3_paths.items():
        print(f"  saved: {rp} ({rp.stat().st_size:,} bytes)")


### Judge the 3-agent ablation reports (same v2 rubrics, same evaluator prompts)

Loops the three reports through `rubric_s{1,2,3}_v2.yaml` with the `S{n}_JUDGE_SYSTEM_PROMPT` variables from the Evaluator prompts cell. The only addition vs the seed-study judge cells is a Condition Note in the user prompt stating that missing sections are absent by design.

In [ ]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# Judges the 3-agent ablation reports with the SAME v2 rubrics and the
# SAME editable system prompts used for the seed-study reports. The only
# addition is a condition note in the user prompt so the judge knows the
# missing sections are absent by design, not truncated.

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "evaluation_refined").is_dir()
)
for p in (PROJECT_ROOT / "evaluation_refined", PROJECT_ROOT / "scripts", PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import run_refined_evaluation as rre

from openai import OpenAI

ABL_REPORT_DIR = PROJECT_ROOT / "evaluation_trial/ablations"
ABL_OUT_DIR = ABL_REPORT_DIR / "seed101_3agent_eval"
ABL_JUDGE_MODEL = "gpt-5.6-sol"
ABL_CONDITION = "3agent"
ABL_API_KEY_FILE = Path.home() / "Github/api_key.env"

ABL_CONDITION_NOTE = (
    "This report comes from an ABLATION condition of the pipeline: three "
    "sequential LLM calls with no RAG retrieval and no knowledge graph. "
    "Sections marked 'not produced in this condition' are absent by design, "
    "not omitted from the report. Score what the report demonstrates; a "
    "capability whose object is entirely absent was not exercised by this "
    "condition."
)

ABL_QUERY_LINE = (
    "Find a substitute material for PFAS material THV (elastomeric terpolymer "
    "of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) "
    "used in the application of broad chemical resistance."
)

abl_api_key = os.environ.get("OPENAI_API_KEY")
if not abl_api_key and ABL_API_KEY_FILE.exists():
    for line in open(ABL_API_KEY_FILE):
        if line.strip().startswith("OPENAI_API_KEY="):
            abl_api_key = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
assert abl_api_key, "no API key: set OPENAI_API_KEY or ABL_API_KEY_FILE"
abl_client = OpenAI(api_key=abl_api_key)

ABL_SYSTEMS = [
    (1, S1_JUDGE_SYSTEM_PROMPT),
    (2, S2_JUDGE_SYSTEM_PROMPT),
    (3, S3_JUDGE_SYSTEM_PROMPT),
]

ABL_OUT_DIR.mkdir(parents=True, exist_ok=True)
abl_results = {}

for sysno, judge_system_prompt in ABL_SYSTEMS:
    rubric_path = PROJECT_ROOT / f"evaluation_refined/rubric_s{sysno}_v2.yaml"
    report_path = ABL_REPORT_DIR / f"{ABL_CONDITION}_system{sysno}_report.md"
    run_label = f"seed101_{ABL_CONDITION}_s{sysno}_v2"

    rubric = rre.load_rubric(rubric_path)
    dim_keys = list(rubric["dimensions"])
    assert dim_keys, f"rubric {rubric_path.name} has no dimensions"
    for _dk in dim_keys:
        _d = rubric["dimensions"][_dk]
        assert _dk.startswith(f"system{sysno}_") and isinstance(_d, dict) \
            and {"rubric", "short_name", "name", "weight"} <= set(_d), (
            f"rubric dimension {_dk!r} malformed (check YAML indentation)"
        )
    smin = float(rubric.get("score_min", 0))
    smax = float(rubric.get("score_max", 10))
    step = float(rubric.get("score_step", 0.5))

    report_text = report_path.read_text(encoding="utf-8")

    scale_block = "\n".join(
        f"- {line}" for line in (rubric.get("ordinal_scale_lines") or [])
    )
    rubric_text = f"### Score scale ({smin:g}-{smax:g}, step {step:g})\n{scale_block}\n"
    if rubric.get("evaluation_principles"):
        rubric_text += f"\n### Evaluation principles\n{rubric['evaluation_principles']}\n"
    for dk in dim_keys:
        dim = rubric["dimensions"][dk]
        rubric_text += f"\n### {dim['name']} (weight: {dim['weight']})\n"
        if dim.get("primary_question"):
            rubric_text += f"Primary question: {dim['primary_question'].strip()}\n\n"
        rubric_text += f"{dim['rubric']}\n"

    output_block = ""
    if rubric.get("output_instructions"):
        output_block += rubric["output_instructions"].strip() + "\n\n"
    output_block += (
        "Respond with ONLY a JSON object in this exact structure:\n\n"
        f"```\n{rubric.get('output_schema', '').strip()}\n```"
    )

    user_prompt = (
        f"## Material Substitution Query\n\n{ABL_QUERY_LINE}\n\n"
        f"## Condition Note\n\n{ABL_CONDITION_NOTE}\n\n"
        f"## Evaluation Rubric (System {sysno} only)\n{rubric_text}\n\n"
        f"## System {sysno} Report\n\n{report_text}\n\n"
        f"## Required Output Format\n\n{output_block}"
    )

    ts = datetime.utcnow().strftime("%Y%m%d%H%M")
    (ABL_OUT_DIR / f"judge_prompt_s{sysno}_{ts}.txt").write_text(
        "### SYSTEM PROMPT\n" + judge_system_prompt
        + "\n\n### USER PROMPT\n" + user_prompt,
        encoding="utf-8",
    )
    print(f"[S{sysno}] rubric {rubric_path.name} | report {report_path.name} | "
          f"prompt {len(user_prompt):,} chars ... ", end="", flush=True)

    t0 = time.time()
    parsed, meta, deviations = rre.call_judge(
        abl_client, ABL_JUDGE_MODEL, judge_system_prompt, user_prompt, rubric,
    )
    elapsed = time.time() - t0
    print(f"{elapsed:.1f}s, finish={meta.get('finish_reason')}")
    if deviations:
        print(f"     protocol deviations: {'; '.join(deviations)}")
    assert "error" not in parsed, parsed
    (ABL_OUT_DIR / f"judge_raw_s{sysno}_{ts}.json").write_text(
        json.dumps({"response": meta.pop("raw_text", ""), "meta": meta},
                   indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    block = parsed.get("A") if isinstance(parsed.get("A"), dict) else parsed

    def _snap(value, smin=smin, smax=smax, step=step):
        try:
            v = float(value)
        except (TypeError, ValueError):
            return None
        return round(max(smin, min(smax, v)) / step) * step

    scores = {}
    for dk in dim_keys:
        entry = block.get(dk) or {}
        scores[dk] = {
            "score": _snap(entry.get("score")),
            "strengths": entry.get("strengths", []),
            "weaknesses": entry.get("weaknesses", []),
            "rationale": entry.get("rationale", ""),
            "confidence": entry.get("confidence", ""),
        }
    valid = [s["score"] for s in scores.values() if s["score"] is not None]
    mean = round(sum(valid) / len(valid), 2) if valid else None

    result = {
        "study": f"Ablation {ABL_CONDITION}, seed 101: LLM judge on "
                 f"{report_path.name} (rubric v2, 0-10 scale)",
        "condition": ABL_CONDITION,
        "query_name": "Query1",
        "query_sentence": ABL_QUERY_LINE,
        "input_format": f"{report_path.name} (flat outputs + verbatim raw call; "
                        "absent sections marked not produced)",
        "report_path": str(report_path),
        "rubric_path": str(rubric_path),
        "dimensions": dim_keys,
        "scores": {run_label: scores},
        "mean_score": mean,
        "judge_model": ABL_JUDGE_MODEL,
        "protocol_deviation": "; ".join(deviations) if deviations else None,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "judge_elapsed_seconds": round(elapsed, 1),
        "judge_completion": meta,
    }
    outfile = ABL_OUT_DIR / f"eval_{run_label}_{ts}.json"
    outfile.write_text(json.dumps(result, indent=2, ensure_ascii=False, default=str),
                       encoding="utf-8")
    abl_results[sysno] = (rubric, dim_keys, scores, mean, outfile)

print()
for sysno, (rubric, dim_keys, scores, mean, outfile) in abl_results.items():
    hdr = [rubric["dimensions"][dk]["short_name"] for dk in dim_keys]
    label = f"seed101_{ABL_CONDITION}_s{sysno}"
    print(f"{'run':<22}" + "".join(f"{h:<10}" for h in hdr) + f"{'Mean':>7}")
    print(f"{label:<22}"
          + "".join(f"{str(scores[dk]['score']):<10}" for dk in dim_keys)
          + f"{mean:>7}")
    for dk in dim_keys:
        s = scores[dk]
        print(f"  [{rubric['dimensions'][dk]['name']}] score = {s['score']} "
              f"(confidence: {s['confidence']})")
        for tag, items in (("+", s["strengths"]), ("-", s["weaknesses"])):
            for item in items:
                print(f"     {tag} {item}")
        print(f"     rationale: {s['rationale']}")
    print(f"  saved: {outfile}\n")


## Ablations: single LLM call (1agent_rag, 1agent_no_rag), seed 101

One call emits requirements, candidate, and manufacturing assessment together, so the three reports share the same provenance (the single verbatim response). For `1agent_rag` the rerun stores the retrieved context + doc-id manifest (the paper artifacts kept only a char count), so the evidence packet is included in the reports. If the pipeline parser rejected the response as malformed JSON, the builder recovers it with minimal deterministic repairs and states the repairs in the report.

In [ ]:
from pathlib import Path
import json

# ============================================================
# Single-call ablations (1agent_rag, 1agent_no_rag), seed 101.
# One LLM call emits all three pipeline sections at once, so the
# three reports share the same provenance: the single response.
# Sections the condition cannot produce are kept and marked as
# not produced; for 1agent_rag the retrieved context (stored by
# our rerun, unlike the paper artifacts) is included as the
# evidence packet.
# ============================================================

ABL1_RUN_DIR = Path("/home/palsubhadeep/Github/MARS/results_seedstudy/seed_101/Query1")
ABL1_OUT_DIR = Path("/home/palsubhadeep/Github/MARS/evaluation_trial/ablations")
ABL1_CONDITIONS = ["1agent_rag", "1agent_no_rag", "1agent_no_rag_openai"]

NOT_PRODUCED_1A = (
    "*Not produced in this condition (single LLM call{extra}). "
    "This section exists in the full MARS pipeline reports.*"
)


def _fmt_recipe_step(step):
    if not isinstance(step, dict):
        return [f"- {step}"]
    idx = step.get("step_index", "?")
    name = step.get("process_step") or step.get("step") or step.get("name") or ""
    lines = [f"**Step {idx}. {name}**"]
    for key, val in step.items():
        if key in ("step_index", "process_step", "step", "name"):
            continue
        if isinstance(val, dict):
            lines.append(f"- {key}:")
            lines += [f"  - {k}: {v}" for k, v in val.items()]
        elif isinstance(val, list):
            lines.append(f"- {key}: " + "; ".join(str(v) for v in val))
        else:
            lines.append(f"- {key}: {val}")
    return lines


def lenient_parse(text):
    """Repair the two local-model JSON defects seen in these runs: a missing
    closing quote before a '}...' run, and missing closing braces at the end.
    Returns (parsed_or_None, list of repair notes)."""
    a, b = text.find("{"), text.rfind("}")
    if a == -1 or b <= a:
        return None, ["no JSON object found"]
    s = text[a : b + 1]
    repairs = []
    for _ in range(6):
        try:
            return json.loads(s), repairs
        except json.JSONDecodeError as e:
            if e.pos >= len(s) - 1:
                missing = s.count("{") - s.count("}")
                if missing <= 0:
                    return None, repairs + [f"unrepaired: {e.msg} at end"]
                s += "}" * missing
                repairs.append(f"appended {missing} missing closing brace(s)")
            elif e.msg.startswith("Expecting ',' delimiter"):
                # The unterminated string swallows the next key's opening
                # quote; step back over it and the '}],' run to the spot
                # where the closing quote is missing.
                j = e.pos
                if j > 0 and s[j - 1] == '"':
                    j -= 1
                while j > 0 and s[j - 1] in "}],":
                    j -= 1
                if j == e.pos:
                    return None, repairs + [f"unrepaired: {e.msg} at char {e.pos}"]
                s = s[:j] + '"' + s[j:]
                repairs.append(f"inserted missing quote at char {j}")
            else:
                return None, repairs + [f"unrepaired: {e.msg} at char {e.pos}"]
    return None, repairs + ["gave up after 6 repair attempts"]


def build_1agent_reports(condition, run_dir, out_dir):
    with open(run_dir / f"ablation_{condition}.json", encoding="utf-8") as f:
        abl = json.load(f)

    has_context = condition == "1agent_rag"
    meta = abl.get("metadata", {})
    query = abl.get("query", {})
    props = (abl.get("required_material_properties") or {}).get("properties", [])
    cons = (abl.get("required_material_properties") or {}).get("constraints", [])
    cand = (abl.get("candidate_selection") or {}).get("final_candidate") or {}
    rejected = (abl.get("candidate_selection") or {}).get("rejected_candidates", [])
    mfg = abl.get("manufacturing_process") or {}
    raw = abl.get("raw_responses") or {}
    single = (raw.get("single_agent") or "").strip()
    context = raw.get("retrieved_context")
    manifest = raw.get("retrieval_manifest") or {}

    # The pipeline's own parser stores empty sections when the single call
    # emits malformed JSON. Recover deterministically from the raw response
    # and say so in the reports; the malformed output is itself a finding.
    parse_note = None
    if single and not props and not cons and not cand:
        reparsed, repairs = lenient_parse(single)
        if reparsed:
            rmp = reparsed.get("required_material_properties") or {}
            props = rmp.get("properties", [])
            cons = rmp.get("constraints", [])
            cs = reparsed.get("candidate_selection") or {}
            cand = cs.get("final_candidate", cs) or {}
            mfg = reparsed.get("manufacturing_process") or mfg
            parse_note = (
                "> **Parse note:** the single call emitted MALFORMED JSON, which "
                "the pipeline's own parser rejected (the run artifact stores "
                "empty sections). The structured content below was recovered "
                "from the verbatim response with minimal deterministic repairs: "
                + "; ".join(repairs) + ". No content was added or altered."
            )
        else:
            parse_note = (
                "> **Parse note:** the single call emitted malformed JSON that "
                "could not be repaired (" + "; ".join(repairs) + "). Structured "
                "sections below are empty; the verbatim response at the end is "
                "the only output."
            )

    not_produced = NOT_PRODUCED_1A.format(
        extra=" with pre-retrieved context" if has_context else ", no RAG, no KG"
    )
    condition_note = (
        f"> **Condition:** `{meta.get('ablation_condition')}` ablation, ONE LLM call "
        f"that emitted the requirement set, the candidate, and the manufacturing "
        f"assessment together"
        + (", after RAG + KG context was pre-retrieved with the raw query and "
           "pasted into the prompt. " if has_context else ", from parametric knowledge only. ")
        + f"Run `{meta.get('pipeline_run_id')}`, duration "
        f"{meta.get('duration_seconds', 0):.0f}s"
        + (f", model {meta['openai_model']} (OpenAI API, temperature 1, "
           "not seeded)" if meta.get("openai_model") else "")
        + ". "
        f"All three system reports for this condition share the same provenance: "
        f"the single verbatim response at the end of each report. Sections marked "
        f"*not produced* are absent by design."
    )
    if parse_note:
        condition_note += "\n>\n" + parse_note

    shared_provenance = [
        "## Agent response (the single call, verbatim, shared by all three reports)",
        "",
        "```",
        single,
        "```",
    ]

    evidence_section = []
    if has_context:
        rag_man = manifest.get("rag") or {}
        kg_man = manifest.get("kg") or {}
        evidence_section = [
            "## Pre-retrieved evidence packet (given to the single call)",
            "",
            "Retrieved with the raw query before the call; the model saw this text "
            "inside its prompt.",
            "",
        ]
        for src, ids in rag_man.items():
            shown = ", ".join(str(i) for i in ids) if ids else "(retrieval failed)"
            evidence_section.append(f"- RAG [{src}]: {len(ids or [])} documents: {shown}")
        for kg, info in kg_man.items():
            if info:
                evidence_section.append(
                    f"- KG [{kg}]: {info.get('matched_nodes')} matched nodes, "
                    f"{info.get('found_paths')} paths"
                )
            else:
                evidence_section.append(f"- KG [{kg}]: search failed")
        if manifest.get("material_db_entries"):
            evidence_section.append(
                f"- Material database listing: {manifest['material_db_entries']} entries"
            )
        evidence_section += [
            "",
            "### Full context text (verbatim)",
            "",
            "```",
            (context or "").strip(),
            "```",
        ]
    elif condition == "1agent_rag":
        evidence_section = [
            "## Pre-retrieved evidence packet",
            "",
            "*Retrieval ran but this artifact stored only its size "
            f"({raw.get('retrieved_context_chars')} chars), not the text.*",
        ]

    # ---------------- System 1 analog ----------------
    s1 = [
        f"# System 1 report ({condition} ablation): requirement extraction",
        "",
        condition_note,
        "",
        "## 1. The query",
        "",
        query.get("sentence", ""),
        "",
        f"- Material X: {query.get('material_X', '')}",
        f"- Application Y: {query.get('application_Y', '')}",
        "",
        "## 2. Initial RAG retrieval",
        "",
    ]
    s1 += (["*See the evidence packet section below; retrieval used the raw query "
            "once, not requirement-specific questions.*"] if has_context
           else [not_produced])
    s1 += [
        "",
        "## 3. Generated questions by LLM",
        "",
        not_produced,
        "",
        "*No dedicated requirement-extraction call exists either: the requirements "
        "below were generated in the same single response as the candidate and the "
        "manufacturing verdict.*",
        "",
        "## 4. Derived outputs by LLM",
        "",
        "### 4.1 Derived material properties",
        "",
    ]
    s1 += [f"{i}. {p}" for i, p in enumerate(props, 1)] or ["(none)"]
    s1 += ["", "### 4.2 Derived hard constraints", ""]
    s1 += [f"{i}. {c}" for i, c in enumerate(cons, 1)] or ["(none)"]
    s1 += [""] + evidence_section + [""] + shared_provenance

    # ---------------- System 2 analog ----------------
    s2 = [
        f"# System 2 report ({condition} ablation): candidate selection",
        "",
        condition_note,
        "",
        "## 1. The extracted properties and hard constraints from System 1",
        "",
        "*Caution: there was no System 1 handoff. These requirements come from the "
        "same single response as the candidate itself, so they cannot be treated as "
        "an independent input that guided selection.*",
        "",
        "### Properties",
        "",
    ]
    s2 += [f"{i}. {p}" for i, p in enumerate(props, 1)] or ["(none)"]
    s2 += ["", "### Hard constraints", ""]
    s2 += [f"{i}. {c}" for i, c in enumerate(cons, 1)] or ["(none)"]
    s2 += [
        "",
        "## 2. KG call to nodes related to material properties",
        "",
    ]
    s2 += (["*No requirement-driven KG mapping; a raw-query KG search was part of "
            "the pre-retrieved packet below.*"] if has_context else [not_produced])
    s2 += [
        "",
        "## 3. Path-node classification (material or not)",
        "",
        not_produced,
        "",
        "## 4. Whole-subgraph classification",
        "",
        not_produced,
        "",
        "## 5. Candidate material proposal and verification",
        "",
        f"*Exactly one proposal, produced inside the single response; no PFAS check, "
        f"no validation questions, no feasibility verdict, no rejection loop. "
        f"Rejected candidates recorded: {len(rejected)}.*",
        "",
        "### 5.1 Proposed candidate",
        "",
        f"- Material name: {cand.get('material_name', '')}",
        f"- Material class: {cand.get('material_class', '')}",
        "",
        "### 5.2 Justification (verbatim)",
        "",
        cand.get("justification", "") or "(none)",
        "",
        "### 5.3 PFAS compliance check",
        "",
        not_produced,
        "",
        "### 5.4 Validation questions and retrieved-evidence answers",
        "",
        not_produced,
        "",
        "### 5.5 Feasibility verdict",
        "",
        not_produced,
        "",
    ]
    s2 += evidence_section + [""] + shared_provenance

    # ---------------- System 3 analog ----------------
    recipe = mfg.get("process_recipe") or []
    s3 = [
        f"# System 3 report ({condition} ablation): manufacturability",
        "",
        condition_note,
        "",
        "## 1. Proposed candidate and justification from System 2",
        "",
        "*Caution: there was no System 2 handoff. The candidate, its justification, "
        "and the manufacturing assessment below come from the same single response.*",
        "",
        f"- Candidate material: {cand.get('material_name', '')}",
        f"- Material class: {cand.get('material_class', '')}",
        "",
        "Justification:",
        "",
        cand.get("justification", "") or "(none)",
        "",
        "## 2. Single vs composite decomposition",
        "",
        not_produced,
        "",
        "## 3. Manufacturing process queries with retrieved evidence",
        "",
    ]
    s3 += (["*No process-specific queries; the only retrieval was the raw-query "
            "packet below, shared by all sections.*"] if has_context
           else [not_produced])
    s3 += [
        "",
        "## 4. Feasibility questions and evidence-based answers",
        "",
        not_produced,
        "",
        "## 5. Feasibility verdict",
        "",
        f"- Status: **{mfg.get('status', 'unknown')}**",
        f"- Blocking constraints: {mfg.get('blocking_constraints') or '(none listed)'}",
        f"- Feedback to System 2: {mfg.get('feedback_to_system2') or '(none)'}",
        "",
        "*No verdict reasoning or confidence exists in this condition; the status "
        "above is the entire verdict.*",
        "",
        f"## 6. Process recipe ({len(recipe)} steps)",
        "",
    ]
    for step in recipe:
        s3 += _fmt_recipe_step(step)
        s3.append("")
    s3 += [
        "### Backing evidence",
        "",
    ]
    s3 += (["*No per-step evidence; only the shared raw-query packet below.*"]
           if has_context else [not_produced])
    s3 += [""] + evidence_section + [""] + shared_provenance

    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for n, lines in ((1, s1), (2, s2), (3, s3)):
        path = out_dir / f"{condition}_system{n}_report.md"
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        paths[n] = path
    summary = {
        "run_id": meta.get("pipeline_run_id"),
        "duration": meta.get("duration_seconds", 0),
        "properties": len(props),
        "constraints": len(cons),
        "candidate": cand.get("material_name"),
        "status": mfg.get("status"),
        "recipe_steps": len(mfg.get("process_recipe") or []),
        "parse_note": bool(parse_note),
    }
    return paths, summary


for _cond in ABL1_CONDITIONS:
    _paths, _s = build_1agent_reports(_cond, ABL1_RUN_DIR, ABL1_OUT_DIR)
    print(f"[{_cond}] run {_s['run_id']} | {_s['duration']:.0f}s"
          + ("  (recovered from malformed JSON)" if _s["parse_note"] else ""))
    print(f"  properties: {_s['properties']} | constraints: {_s['constraints']} | "
          f"candidate: {_s['candidate']}")
    print(f"  mfg status: {_s['status']} | recipe steps: {_s['recipe_steps']}")
    for n, p in _paths.items():
        print(f"  saved: {p} ({p.stat().st_size:,} bytes)")


### Judge the single-call ablation reports (same v2 rubrics, same evaluator prompts)

Loops both conditions x three reports through `rubric_s{1,2,3}_v2.yaml` with the `S{n}_JUDGE_SYSTEM_PROMPT` variables from the Evaluator prompts cell, plus a per-condition note stating that missing sections are absent by design.

In [ ]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# Judges the single-call ablation reports (1agent_rag, 1agent_no_rag)
# with the SAME v2 rubrics and the SAME editable system prompts. The only
# addition is a per-condition note in the user prompt so the judge knows
# missing sections are absent by design, not truncated.

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "evaluation_refined").is_dir()
)
for p in (PROJECT_ROOT / "evaluation_refined", PROJECT_ROOT / "scripts", PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import run_refined_evaluation as rre

from openai import OpenAI

A1_REPORT_DIR = PROJECT_ROOT / "evaluation_trial/ablations"
A1_JUDGE_MODEL = "gpt-5.6-sol"
A1_API_KEY_FILE = Path.home() / "Github/api_key.env"

A1_CONDITION_NOTES = {
    "1agent_rag": (
        "This report comes from an ABLATION condition of the pipeline: ONE LLM "
        "call that produced the requirement set, the candidate, and the "
        "manufacturing assessment together, with RAG and KG context "
        "pre-retrieved using the raw query and pasted into the prompt. The "
        "retrieved evidence packet is included in the report. Sections marked "
        "'not produced in this condition' are absent by design, not omitted. "
        "Score what the report demonstrates; a capability whose object is "
        "entirely absent was not exercised by this condition."
    ),
    "1agent_no_rag": (
        "This report comes from an ABLATION condition of the pipeline: ONE LLM "
        "call, purely from parametric knowledge, that produced the requirement "
        "set, the candidate, and the manufacturing assessment together. No "
        "retrieval and no knowledge graph were available. Sections marked 'not "
        "produced in this condition' are absent by design, not omitted. Score "
        "what the report demonstrates; a capability whose object is entirely "
        "absent was not exercised by this condition."
    ),
    "1agent_no_rag_openai": (
        "This report comes from an ABLATION condition of the pipeline: ONE LLM "
        "call, purely from parametric knowledge, that produced the requirement "
        "set, the candidate, and the manufacturing assessment together. No "
        "retrieval and no knowledge graph were available. Sections marked 'not "
        "produced in this condition' are absent by design, not omitted. Score "
        "what the report demonstrates; a capability whose object is entirely "
        "absent was not exercised by this condition."
    ),
    "3agent_rag": (
        "This report comes from an ABLATION condition of the pipeline: three "
        "sequential LLM calls (requirement extraction, candidate selection, "
        "manufacturability) chained by parsed handoffs, each given the SAME "
        "pre-retrieved RAG+KG evidence packet, fetched once by the harness "
        "with the raw query; no agent directed any retrieval and no "
        "validation or iteration loop ran. Sections marked 'not produced in "
        "this condition' are absent by design, not omitted. Score what the "
        "report demonstrates; a capability whose object is entirely absent "
        "was not exercised by this condition."
    ),
}

A1_QUERY_LINE = (
    "Find a substitute material for PFAS material THV (elastomeric terpolymer "
    "of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) "
    "used in the application of broad chemical resistance."
)

a1_api_key = os.environ.get("OPENAI_API_KEY")
if not a1_api_key and A1_API_KEY_FILE.exists():
    for line in open(A1_API_KEY_FILE):
        if line.strip().startswith("OPENAI_API_KEY="):
            a1_api_key = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
assert a1_api_key, "no API key: set OPENAI_API_KEY or A1_API_KEY_FILE"
a1_client = OpenAI(api_key=a1_api_key)

A1_SYSTEM_PROMPTS = {
    1: S1_JUDGE_SYSTEM_PROMPT,
    2: S2_JUDGE_SYSTEM_PROMPT,
    3: S3_JUDGE_SYSTEM_PROMPT,
}

a1_all_results = {}

for condition, condition_note in A1_CONDITION_NOTES.items():
    out_dir = A1_REPORT_DIR / f"seed101_{condition}_eval"
    out_dir.mkdir(parents=True, exist_ok=True)

    for sysno, judge_system_prompt in A1_SYSTEM_PROMPTS.items():
        rubric_path = PROJECT_ROOT / f"evaluation_refined/rubric_s{sysno}_v2.yaml"
        report_path = A1_REPORT_DIR / f"{condition}_system{sysno}_report.md"
        run_label = f"seed101_{condition}_s{sysno}_v2"

        rubric = rre.load_rubric(rubric_path)
        dim_keys = list(rubric["dimensions"])
        assert dim_keys, f"rubric {rubric_path.name} has no dimensions"
        for _dk in dim_keys:
            _d = rubric["dimensions"][_dk]
            assert _dk.startswith(f"system{sysno}_") and isinstance(_d, dict) \
                and {"rubric", "short_name", "name", "weight"} <= set(_d), (
                f"rubric dimension {_dk!r} malformed (check YAML indentation)"
            )
        smin = float(rubric.get("score_min", 0))
        smax = float(rubric.get("score_max", 10))
        step = float(rubric.get("score_step", 0.5))

        report_text = report_path.read_text(encoding="utf-8")

        scale_block = "\n".join(
            f"- {line}" for line in (rubric.get("ordinal_scale_lines") or [])
        )
        rubric_text = f"### Score scale ({smin:g}-{smax:g}, step {step:g})\n{scale_block}\n"
        if rubric.get("evaluation_principles"):
            rubric_text += f"\n### Evaluation principles\n{rubric['evaluation_principles']}\n"
        for dk in dim_keys:
            dim = rubric["dimensions"][dk]
            rubric_text += f"\n### {dim['name']} (weight: {dim['weight']})\n"
            if dim.get("primary_question"):
                rubric_text += f"Primary question: {dim['primary_question'].strip()}\n\n"
            rubric_text += f"{dim['rubric']}\n"

        output_block = ""
        if rubric.get("output_instructions"):
            output_block += rubric["output_instructions"].strip() + "\n\n"
        output_block += (
            "Respond with ONLY a JSON object in this exact structure:\n\n"
            f"```\n{rubric.get('output_schema', '').strip()}\n```"
        )

        user_prompt = (
            f"## Material Substitution Query\n\n{A1_QUERY_LINE}\n\n"
            f"## Condition Note\n\n{condition_note}\n\n"
            f"## Evaluation Rubric (System {sysno} only)\n{rubric_text}\n\n"
            f"## System {sysno} Report\n\n{report_text}\n\n"
            f"## Required Output Format\n\n{output_block}"
        )

        ts = datetime.utcnow().strftime("%Y%m%d%H%M")
        (out_dir / f"judge_prompt_s{sysno}_{ts}.txt").write_text(
            "### SYSTEM PROMPT\n" + judge_system_prompt
            + "\n\n### USER PROMPT\n" + user_prompt,
            encoding="utf-8",
        )
        print(f"[{condition} S{sysno}] prompt {len(user_prompt):,} chars ... ",
              end="", flush=True)

        t0 = time.time()
        parsed, meta, deviations = rre.call_judge(
            a1_client, A1_JUDGE_MODEL, judge_system_prompt, user_prompt, rubric,
        )
        elapsed = time.time() - t0
        print(f"{elapsed:.1f}s, finish={meta.get('finish_reason')}")
        if deviations:
            print(f"     protocol deviations: {'; '.join(deviations)}")
        assert "error" not in parsed, parsed
        (out_dir / f"judge_raw_s{sysno}_{ts}.json").write_text(
            json.dumps({"response": meta.pop("raw_text", ""), "meta": meta},
                       indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        block = parsed.get("A") if isinstance(parsed.get("A"), dict) else parsed

        def _snap(value, smin=smin, smax=smax, step=step):
            try:
                v = float(value)
            except (TypeError, ValueError):
                return None
            return round(max(smin, min(smax, v)) / step) * step

        scores = {}
        for dk in dim_keys:
            entry = block.get(dk) or {}
            scores[dk] = {
                "score": _snap(entry.get("score")),
                "strengths": entry.get("strengths", []),
                "weaknesses": entry.get("weaknesses", []),
                "rationale": entry.get("rationale", ""),
                "confidence": entry.get("confidence", ""),
            }
        valid = [s["score"] for s in scores.values() if s["score"] is not None]
        mean = round(sum(valid) / len(valid), 2) if valid else None

        result = {
            "study": f"Ablation {condition}, seed 101: LLM judge on "
                     f"{report_path.name} (rubric v2, 0-10 scale)",
            "condition": condition,
            "query_name": "Query1",
            "query_sentence": A1_QUERY_LINE,
            "input_format": f"{report_path.name} (flat outputs + verbatim single "
                            "call; absent sections marked not produced)",
            "report_path": str(report_path),
            "rubric_path": str(rubric_path),
            "dimensions": dim_keys,
            "scores": {run_label: scores},
            "mean_score": mean,
            "judge_model": A1_JUDGE_MODEL,
            "protocol_deviation": "; ".join(deviations) if deviations else None,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "judge_elapsed_seconds": round(elapsed, 1),
            "judge_completion": meta,
        }
        outfile = out_dir / f"eval_{run_label}_{ts}.json"
        outfile.write_text(
            json.dumps(result, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8",
        )
        a1_all_results[(condition, sysno)] = (rubric, dim_keys, scores, mean, outfile)

print()
for (condition, sysno), (rubric, dim_keys, scores, mean, outfile) in a1_all_results.items():
    hdr = [rubric["dimensions"][dk]["short_name"] for dk in dim_keys]
    label = f"{condition}_s{sysno}"
    print(f"{'run':<24}" + "".join(f"{h:<10}" for h in hdr) + f"{'Mean':>7}")
    print(f"{label:<24}"
          + "".join(f"{str(scores[dk]['score']):<10}" for dk in dim_keys)
          + f"{mean:>7}")
    for dk in dim_keys:
        s = scores[dk]
        print(f"  [{rubric['dimensions'][dk]['name']}] score = {s['score']} "
              f"(confidence: {s['confidence']})")
        for tag, items in (("+", s["strengths"]), ("-", s["weaknesses"])):
            for item in items:
                print(f"     {tag} {item}")
        print(f"     rationale: {s['rationale']}")
    print(f"  saved: {outfile}\n")
